# AKE Lab 4 · Moderne NLP & LLM-Anpassung — Starter-Notebook

**Angewandte KI & KI-Engineering · Block B – Spezialisierte Anwendungsgebiete · Woche 4**
Skript: *Moderne NLP & LLM-Anpassung*, Abschn. 1 (Leitbeispiel, Modell), 2 (Decoder, Dekodierung, Prompting), 4 (Instruction-Tuning), 5 (LoRA), 7–8 (Evaluation)

> **Dieses Notebook ist das komplette Lab – es gibt kein separates Aufgabenblatt.** Jede Aufgabe enthält oben die Aufgabenstellung,
> alle benötigten Formeln, die Erwartungswerte und die Leitfragen. Der Code ist ausführlich kommentiert: Lesen Sie die Kommentare
> (Zeilen, die mit `#` beginnen) – sie erklären Zeile für Zeile, was passiert.

Sie passen ein kleines Sprachmodell an eine eng geschnittene Aufgabe an – das Leitbeispiel des Skripts: Aus einer frei formulierten
**Störmeldung** soll ein **JSON-Objekt** mit den Feldern `anlage`, `symptom`, `messwert`, `ursache`, `massnahme` werden. Ein Beispiel:

| Eingabe (Störmeldung) | gewünschte Ausgabe (JSON) |
|---|---|
| *Presse 3 steht seit 06:40, der Hydraulikdruck fällt ab …* | `{"anlage": "Presse 3", "symptom": "…", "messwert": "…", "ursache": "…", "massnahme": "…"}` |

Sie messen erst, was das unveränderte Modell kann (Prompting-Baseline), trainieren dann einen **LoRA-Adapter** auf 300 Beispielen
und weisen nach, ob und wie stark er besser ist – mit gepaartem Vergleich und einem Regressions-Set gegen Vergessen. Es ist dasselbe
Training wie in der Live-Demo der Vorlesung, diesmal auf Ihrem Rechner.

**Charakter:** Übung in **Blöcken während der Vorlesung** – jeder Block folgt direkt auf den Vorlesungsabschnitt, den er vertieft
(Fahrplan unten). Die Zellen **`PRINZIP`** zeigen mit bewusst einfachem Code – Schleifen, `print`, keine Hilfsfunktionen –, was in
Modell, Dekodierung, Verlust und LoRA *intern* passiert, und prüfen am Ende, dass die Bibliothek genau dasselbe rechnet. Dazu kommen
kurze `CODE`-Lücken (je 1–2 Zeilen) und Interpretationsfragen. Aufgabe 7 und die optionalen Teile sind Nachbereitung.
**Ihre Zeit ≈ 70 Min.** in der Sitzung, davon **≈ 10–12 Min. reine Rechenzeit** auf der Referenz (4 Kerne / 8 GB, keine GPU);
das Training (≈ 4–7 Min.) läuft, während die Vorlesung weitergeht.

**Vorab, einmalig (mit Internet):** `setup_check.ipynb` in diesem Ordner ausführen. Die Gewichte von SmolLM2-135M-Instruct
(≈ 270 MB) sind zu groß für GitHub und liegen nicht im Repository; `setup_check.ipynb` – oder notfalls die erste Code-Zelle dieses
Notebooks – lädt sie einmalig nach `data/models/SmolLM2-135M-Instruct/`. Danach arbeitet das Lab **offline**.

**Fallback:** Jeder Schritt schreibt ein JSON-Log nach `results/`. Dauert das Training zu lange, setzen Sie in Aufgabe 4
`NUTZE_REFERENZ_ADAPTER = True`: Dann wird der mitgelieferte Adapter aus `results_referenz/adapter_stoermeldung/` geladen (1,8 MB) –
alle späteren Aufgaben laufen damit genauso.

## Organisatorisches

| | |
|---|---|
| **Termin** | Woche 4 (KW 44, 29.10.2026), in Blöcken während der Vorlesung; Aufgabe 7 und Optionales als Nachbereitung, Besprechung 5 Min. zu Beginn der Sitzung in Woche 5 |
| **Skript-Bezug** | *Moderne NLP & LLM-Anpassung*: Bsp. 1.1 (Leitbeispiel), Abschn. 1.4 (Modell), 2.1–2.3 (Decoder, Perplexität, Dekodierung, Prompting-Baseline), 4.1 (Chat-Template, Loss-Maske), 5.3–5.8 (LoRA, Def. 5.2, Bsp. 5.3, 5.4, Listings), 7.5–7.6 (Judge-Kosten, Unsicherheit, gepaarter Vergleich), 8.1–8.2 (Eval-Setup, regelbasierte Stufen, Fehleranalyse) |
| **Material** | Ordner `lab04/` im Kursrepository: dieses Notebook, `setup_check.ipynb`, `results_referenz/` (Referenzlogs und trainierter Adapter, Fallback); die Musterlösung wird nach der Besprechung freigeschaltet |
| **Umgebung** | Kursumgebung (`git pull`, danach `uv sync`); `peft` ist bereits enthalten. **Vorab mit Internet:** `setup_check.ipynb` (oder die erste Code-Zelle in Aufgabe 0) lädt einmalig die Gewichte von SmolLM2-135M-Instruct (nicht im Repository, zu groß für GitHub); danach kein Hub-Zugriff |
| **Daten** | `data/stoermeldungen/`: `train.jsonl` (300), `val.jsonl` (30), `eval.jsonl` (50, andere Anlagen als im Training, geschichtet: 30 typisch · 10 mit fehlenden Angaben · 10 mit Abkürzungen/Tippfehlern), `regression.jsonl` (10 allgemeine Anweisungen). Synthetisch erzeugt, Seed 0 |

**Fahrplan**

| Aufgabe | Block in der Vorlesung (nach Folie …) | Inhalt | Ihre Zeit | Rechenzeit (4 Kerne) |
|---|---|---|---|---|
| 0 · Setup | vorab, spätestens zu Beginn | Setup-Zellen, Prüfungen | 3 Min. | Sekunden |
| 1 · Das Modell von innen | „Leitbeispiel und Modell“ | `PRINZIP` Bauplan, Parameter von Hand (`CODE` 1 Zeile) | 7 Min. | Sekunden |
| 2.1–2.2 · Decoder von Hand (1/2) | „Beispiel: Perplexität von Hand“ | `PRINZIP` nächstes Token, Perplexität (`CODE` 1 Zeile) | 8 Min. | Sekunden |
| 2.3–2.6 · Decoder von Hand (2/2) | „Zwischenstand: Decoder“ | `PRINZIP` Temperatur/Top-p, Greedy-Schleife (`CODE` je 1 Zeile); Prompting-Baseline (`CODE` 1 Zeile) | 16 Min. | ≈ 1–3 Min. |
| 3 · Instruction-Tuning | „Zwischenstand: Instruction-Tuning“ | `PRINZIP` Chat-Template, Loss-Maske (`CODE` 1 Zeile), Verlust von Hand (`CODE` 2 Zeilen) | 10 Min. | Sekunden |
| 4 · LoRA-Training starten | „Demo: LoRA-Training starten“ | Konfiguration (`CODE` 1 Zeile), Training läuft im Hintergrund weiter | 3 Min. | ≈ 4–7 Min. |
| 5 · LoRA von innen | „Zwischenstand: LoRA“ | `PRINZIP` Rang, Rang-1-Update, eigene LoRA-Schicht (`CODE` je 1 Zeile), PEFT nachgerechnet (`PAPIER`) | 13 Min. | Sekunden |
| 6 · Evaluation vorher/nachher | direkt im Anschluss an Aufgabe 5 | Metriken, gepaarter Vorzeichentest (`CODE` 2 Zeilen), Fehler lesen | 9 Min. | ≈ 1 Min. |
| 7 · Regressions-Set | Nachbereitung | Vergessen prüfen | 4 Min. | ≈ 1 Min. |
| Optional | Nachbereitung | O1 Judge-Kosten · O2 alle Zielmodule | – | O2 ≈ 10 Min. |

**Kennzeichnung der Arbeitsarten** (steht am Anfang jeder Aufgabe):

| Marke | Was Sie tun |
|---|---|
| `PRINZIP` | ausführen und lesen – bewusst einfacher Code, der ein Prinzip erklärt und am Ende mit der Bibliothek vergleicht; oft mit einer kleinen Lücke |
| `CODE` | eine Lücke selbst füllen (Zeile mit `raise NotImplementedError` durch Ihre Lösung ersetzen) |
| `PAPIER` | erst von Hand rechnen, dann mit dem Code vergleichen |
| `AUSFÜHREN` | nur ausführen und die Ausgabe lesen |
| `DENKEN` | Interpretationsfrage in der Markdown-Zelle beantworten |

Wer bei einer Lücke nicht weiterkommt, übernimmt die Zeile aus der Lösung – die Aufgaben bauen aufeinander auf.

**Abgabe:** keine. Bringen Sie zur Besprechung in Woche 5 Ihre Tabelle aus 6.1 und Ihre Antworten auf die Interpretationsfragen zu
Aufgabe 6 und 7 mit. **Aufbewahren:** den Adapter `results/adapter_stoermeldung/` – er ist das erste versionierte Modellartefakt in
*Daten-Engineering für ML*.

## Lernziele

Nach dem Lab können Sie …

- **Verstehen:** den Bauplan eines Decoder-Sprachmodells benennen, die Dekodierung (Softmax, Temperatur, Top-p, greedy) und die
  LoRA-Formel $\mathbf h = \mathbf W_0\mathbf x + \tfrac{\alpha}{r}\,\mathbf B\mathbf A\mathbf x$ im Code nachvollziehen.
- **Verstehen:** erklären, wie ein Instruction-Tuning-Beispiel aufgebaut ist und warum der Verlust nur auf der Antwort berechnet wird.
- **Anwenden:** ein LoRA-Training durchführen und die Zahl der trainierbaren Parameter vorab berechnen.
- **Analysieren:** Eval-Ergebnisse gestuft, mit Intervall und gepaartem Vergleich berichten; Fehler lesen und kategorisieren.

## Spielregeln und Hinweise

> **In der Praxis: Erst das Eval-Setup, dann das Training.** Bevor ein einziger Gradientenschritt läuft, steht fest, *woran* gemessen
> wird (Format, Schema, exakte Felder), *womit* verglichen wird (Prompting-Baseline) und *wie* Unsicherheit berichtet wird
> (gepaarter Vergleich, Intervall). Genau in dieser Reihenfolge ist das Notebook aufgebaut (Skript Abschn. 8.1).

- **Reproduzierbarkeit:** `torch.manual_seed(0)` vor jedem Modell und Training; Dekodierung immer **greedy** mit `max_new_tokens=96`
  (Skript Abschn. 2.2: Dekodierung gehört zum Versuch). Jeder Schritt schreibt sein JSON-Log nach `results/`.
- **Eval-Set nur zum Messen:** `eval.jsonl` wird nie zum Training oder zur Auswahl von Hyperparametern benutzt. Die Konfiguration
  steht vorab fest (Skript Tab. LoRA-Hyperparameter).
- **Offline:** `HF_HUB_OFFLINE=1` wird gesetzt, *bevor* `transformers` importiert wird (erledigt die Setup-Zelle).
  Fehlermeldungen wie `OSError ... not found` heißen: Das Modell liegt nicht unter `data/models/` → erst `setup_check.ipynb`.
- **`PRINZIP`-Zellen sind absichtlich unoptimiert** (z. B. Generierung ohne Zwischenspeicher, Schleife statt Vektoroperation).
  Sie zeigen, *was* gerechnet wird. Die Bibliothek rechnet dasselbe, nur schneller – das vergleicht jede dieser Zellen am Ende.
- **Notbremse:** Mehr als 8 s je Trainingsschritt in Aufgabe 4 → abbrechen (■) und den Referenz-Adapter verwenden.
- **Apple Silicon:** `DEVICE = torch.device("mps")` in der Setup-Zelle beschleunigt das Training etwa dreifach; die letzten
  Nachkommastellen können dann abweichen.
- **Erwartungswerte** stammen aus dem Lösungslauf (Seed 0, 4 Threads). Die Werte nach dem Training sollten Sie auf wenige
  Prozentpunkte treffen; kleine Abweichungen durch andere CPU-Bibliotheken sind normal.

## Kurzhilfe: Notebook und Python lesen

**Bedienung:** Zelle auswählen und mit **Shift + Enter** ausführen. Die Zellen bauen aufeinander auf – immer **von oben nach unten**
ausführen. Nach einem Fehler die Zelle korrigieren und erneut ausführen. Geht gar nichts mehr: *Kernel → Restart* und wieder von oben
beginnen (das Modell wird dann neu geladen). Das Notebook bleibt die ganze Sitzung geöffnet: Das in Aufgabe 1 geladene Modell wird bis
Aufgabe 7 weiterverwendet.

| Sie sehen im Code … | Bedeutung |
|---|---|
| `# Text` | Kommentar – wird nicht ausgeführt, erklärt nur |
| `x = 3` | Zuweisung: Der Name `x` steht ab jetzt für den Wert 3 |
| `def f(a, b): ...` | Definition einer Funktion mit den Eingaben `a` und `b`; `return` gibt das Ergebnis zurück |
| `for t in range(1, 5):` | Schleife: Der eingerückte Block läuft für `t = 1, 2, 3, 4` (die obere Grenze ist **nicht** dabei) |
| `liste[0]`, `liste[-1]`, `liste[2:5]` | erstes Element, letztes Element, Elemente 2 bis 4 (Zählung beginnt bei 0) |
| `[f(b) for b in daten]` | *List Comprehension*: wendet `f` auf jedes Element an und sammelt die Ergebnisse in einer Liste |
| `{"anlage": "Presse 3"}`, `d["anlage"]` | *Dictionary* (Schlüssel → Wert) und Zugriff über den Schlüssel |
| `f"p = {p:.3f}"` | *f-String*: setzt den Wert von `p` mit 3 Nachkommastellen in den Text ein |
| `np.array`, `torch.tensor` | Vektoren/Matrizen (NumPy bzw. PyTorch); `.shape` gibt die Form an, z. B. `(4, 576)` = 4 Zeilen, 576 Spalten |
| `A @ B`, `W.T` | Matrixprodukt und transponierte Matrix |
| `with torch.no_grad():` | Im eingerückten Block merkt sich PyTorch keine Rechenschritte für Gradienten – spart Zeit und Speicher beim reinen Rechnen |
| `float(t)`, `int(t)` | macht aus einem Tensor mit einem einzigen Wert eine normale Python-Zahl |
| `check("Name", lambda: ...)` | Prüfzelle: zeigt ✅, wenn die Bedingung hinter `lambda:` wahr ist, sonst ❌ mit Hinweis |
| `raise NotImplementedError("TODO …")` | Platzhalter in einer Lücke: Diese Zeile ersetzen Sie durch Ihre Lösung |

## Aufgabe 0 · Setup

`AUSFÜHREN` · Pflicht, ohne Abgabe · ≈ 3 Min. · **vorab mit Internet, spätestens zu Beginn der Sitzung**

1. **Vorab (mit Internet):** im Ordner des Repositorys `git pull` und `uv sync`, dann `lab04/setup_check.ipynb` einmal ausführen:
   Paketprüfung, Download und Prüfsumme der Gewichte, Ladetest offline, Hochrechnung der Trainingszeit aus drei Schritten.
   Liegt die Hochrechnung über 20 Min., in Aufgabe 4 den mitgelieferten Adapter verwenden.
2. Jupyter im Ordner des Repositorys starten (`uv run jupyter notebook`), dieses Notebook als `lab04_<nachname>.ipynb` kopieren und nur
   in der Kopie arbeiten.
3. Die drei folgenden Zellen ausführen. **Erwartung:** alle Prüfungen ✅.

**Woher kommt das Modell?** Die Gewichte von SmolLM2-135M-Instruct (`model.safetensors`, ≈ 270 MB) sind zu groß für GitHub (Limit
100 MB je Datei) und liegen deshalb **nicht** im Repository. Die erste Code-Zelle prüft, ob alle Modelldateien unter
`data/models/SmolLM2-135M-Instruct/` liegen, und lädt fehlende Dateien **einmalig** von huggingface.co (fester Versionsstand,
Prüfsumme wird kontrolliert). Hat `setup_check.ipynb` das schon erledigt, passiert nichts – dann braucht die Zelle auch kein Internet.
**Kein Internet oder Hub gesperrt?** `model.safetensors` vom Hochschul-Share (Ordner *Lab04 / models / SmolLM2-135M-Instruct*) in
diesen Ordner kopieren und die Zelle erneut ausführen.

Die Setup-Zelle danach schaltet den Hub-Zugriff ab, lädt die Bibliotheken, begrenzt PyTorch auf höchstens 4 Threads (vergleichbare
Zeitmessungen), definiert zwei Hilfsfunktionen (`check` für die Prüfungen, `abschnitt` für die Laufzeittabelle am Ende), sucht das
Kursrepository und prüft, ob Daten und Modell vorhanden sind.

In [ ]:
# --- Modelldateien holen (nur, wenn sie fehlen) -----------------------------------------------------------------
# Diese Zelle benutzt nur die Standardbibliothek von Python (kein transformers), damit die Setup-Zelle danach den
# Offline-Modus noch vor dem Import von transformers einschalten kann.
import hashlib, ssl, urllib.request
from pathlib import Path
try:                                                    # Zertifikate fuer https: das Python von uv findet die des
    import certifi                                      # Betriebssystems oft nicht, das Paket certifi bringt eigene mit
    _ssl = ssl.create_default_context(cafile=certifi.where())
except ImportError:
    _ssl = ssl.create_default_context()

# Kursrepository suchen: vom aktuellen Ordner (lab04/) nach oben, bis ein Ordner pyproject.toml enthaelt
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if _root is None:
    raise FileNotFoundError("Kein Kursrepository gefunden. Bitte Jupyter im Ordner des Repositorys starten und lab04/ oeffnen.")
MODELL_ORDNER = _root / "data" / "models" / "SmolLM2-135M-Instruct"
# Fester Versionsstand (Commit) auf dem Hub -> alle laden exakt dieselben Dateien wie im Loesungslauf
HUB_URL = "https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct/resolve/12fd25f77366fa6b3b4b768ec3050bf629380bac/"
DATEIEN = ["config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
           "special_tokens_map.json", "merges.txt", "vocab.json", "model.safetensors"]
SHA256_GEWICHTE = "5af571cbf074e6d21a03528d2330792e532ca608f24ac70a143f6b369968ab8c"   # Fingerabdruck der Gewichte

def sha256(pfad):
    """Pruefsumme einer Datei, blockweise gelesen (1 MB je Block), damit nicht alles in den Speicher muss."""
    h = hashlib.sha256()
    with open(pfad, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def herunterladen(name):
    """Laedt eine Datei vom Hub erst in <name>.part und benennt sie erst am Ende um (kein halber Download bleibt liegen)."""
    ziel, teil = MODELL_ORDNER / name, MODELL_ORDNER / (name + ".part")
    with urllib.request.urlopen(HUB_URL + name, context=_ssl) as antwort, open(teil, "wb") as f:
        gesamt, geladen, gemeldet = int(antwort.headers.get("Content-Length", 0)), 0, 0
        while block := antwort.read(1 << 20):           # 1 MB nach dem anderen lesen, bis nichts mehr kommt
            f.write(block); geladen += len(block)
            if gesamt > 50e6 and geladen / gesamt >= gemeldet + 0.1:   # bei grossen Dateien alle 10 % Fortschritt melden
                gemeldet += 0.1; print(f"  {name}: {geladen / 1e6:.0f} von {gesamt / 1e6:.0f} MB")
    teil.rename(ziel)

MODELL_ORDNER.mkdir(parents=True, exist_ok=True)
fehlend = [n for n in DATEIEN if not (MODELL_ORDNER / n).exists()]
if fehlend:
    print(f"Lade {len(fehlend)} Datei(en) von huggingface.co nach {MODELL_ORDNER} (einmalig, ≈ 270 MB, 1–5 Min.) ...")
    try:
        for name in fehlend:
            herunterladen(name)
    except OSError as e:                                # kein Internet, Hub gesperrt, Proxy ...
        raise RuntimeError(f"Download fehlgeschlagen ({e}). Mit Internet erneut ausfuehren oder model.safetensors "
                           "vom Hochschul-Share in den Ordner oben kopieren.") from None
    print("Download fertig.")
else:
    print(f"Alle Modelldateien liegen bereits in {MODELL_ORDNER} – kein Download noetig.")

if sha256(MODELL_ORDNER / "model.safetensors") != SHA256_GEWICHTE:
    raise RuntimeError("model.safetensors ist unvollstaendig oder eine andere Version: Datei loeschen und Zelle erneut ausfuehren.")
print("✅ Pruefsumme der Gewichte stimmt (gleiche Gewichte wie im Loesungslauf)")

In [ ]:
# --- Offline-Modus: MUSS vor dem Import von transformers / peft stehen ----------------------------------------
# Umgebungsvariablen steuern Bibliotheken von aussen. transformers liest sie beim Import - deshalb stehen sie ganz oben.
import os
os.environ["HF_HUB_OFFLINE"] = "1"                       # kein Hub-Zugriff: das Modell liegt in data/models/
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")  # unterdrueckt Warnungen der schnellen Tokenizer

# --- Bibliotheken -------------------------------------------------------------------------------------------
import sys, time, json, math, warnings                  # Standardbibliothek: Zeitmessung, JSON lesen/schreiben, Mathematik
from pathlib import Path                                # Pfade zu Dateien und Ordnern, unabhaengig vom Betriebssystem
from collections import Counter                         # zaehlt, wie oft jedes Element vorkommt
import numpy as np                                      # Rechnen mit Vektoren und Matrizen
import pandas as pd                                     # Tabellen (DataFrame) fuer uebersichtliche Ausgaben
import torch                                            # PyTorch: Tensoren, neuronale Netze, Gradienten
import matplotlib.pyplot as plt                         # Diagramme
warnings.filterwarnings("ignore", message=r".*encountered in matmul", category=RuntimeWarning)  # Scheinwarnung (macOS)
import huggingface_hub, transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM   # Tokenizer und Sprachmodell aus einem Ordner laden
from peft import LoraConfig, get_peft_model, PeftModel         # PEFT: LoRA-Adapter anlegen, laden, speichern
transformers.utils.logging.set_verbosity_error()        # nur echte Fehler von transformers anzeigen
transformers.utils.logging.disable_progress_bar()

torch.set_num_threads(min(4, os.cpu_count()))           # Referenz: 4 Kerne -> vergleichbare Laufzeiten
torch.manual_seed(0); np.random.seed(0)                 # feste Zufallszahlen -> jeder Lauf liefert dasselbe Ergebnis
DEVICE = torch.device("cpu")                            # Apple Silicon: torch.device("mps") ist etwa dreimal schneller

# --- Pruefhelfer --------------------------------------------------------------------------------------------
# check("Name", Bedingung, "Hinweis") gibt ✅ aus, wenn die Bedingung wahr ist, sonst ❌ mit Hinweis.
# Die Bedingung steht meist als `lambda: ...` da: eine kleine Funktion, die erst hier im try-Block ausgewertet wird.
# So fuehrt ein Fehler in Ihrer Loesung zu einem ❌ mit Fehlermeldung statt zum Abbruch der ganzen Zelle.
CHECKS = {}                                             # merkt sich alle Pruefungen fuer die Uebersicht am Ende
def check(name, ok, hint=""):
    """ok: Wahrheitswert oder Funktion ohne Argumente (Fehler darin -> ❌ mit Fehlermeldung)."""
    try:
        ok = bool(ok() if callable(ok) else ok); msg = hint
    except NotImplementedError as e:                    # die Luecke ist noch nicht gefuellt
        ok, msg = False, f"{e} ist noch offen"
    except Exception as e:                              # irgendein anderer Fehler in der Loesung
        ok, msg = False, f"{type(e).__name__}: {e}" + (f" | {hint}" if hint else "")
    CHECKS[name] = ok
    print(f"✅ {name}" if ok else f"❌ {name} – Hinweis: {msg}")

# --- Laufzeitmessung je Aufgabe (Tabelle am Ende) -------------------------------------------------------------
# abschnitt("Aufgabe 2") legt fest, welcher Aufgabe die Rechenzeit der folgenden Zellen zugeordnet wird.
# Jupyter ruft _ake_pre vor und _ake_post nach jeder Zelle auf; die Differenz wird aufsummiert. Fuer das Lab nicht wichtig.
RUNTIME, _teil = {}, {"name": "Aufgabe 0", "t0": None}
def abschnitt(name): _teil["name"] = name
def _ake_pre(*a): _teil["t0"] = time.perf_counter()
def _ake_post(*a):
    if _teil["t0"] is not None:
        RUNTIME[_teil["name"]] = RUNTIME.get(_teil["name"], 0.0) + time.perf_counter() - _teil["t0"]
try:
    _ip = get_ipython()
    for ev, fn in (("pre_run_cell", _ake_pre), ("post_run_cell", _ake_post)):
        for old in [f for f in _ip.events.callbacks[ev] if getattr(f, "__name__", "") == fn.__name__]:
            _ip.events.unregister(ev, old)              # bei erneutem Ausfuehren nicht doppelt registrieren
        _ip.events.register(ev, fn)
except NameError:                                       # ausserhalb von Jupyter gibt es get_ipython nicht
    pass

# --- Pfade: REPO_ROOT = erster Elternordner mit pyproject.toml ------------------------------------------------
# Wir gehen vom aktuellen Ordner (lab04/) nach oben, bis ein Ordner die Datei pyproject.toml enthaelt.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Kein Kursrepository gefunden. Bitte Jupyter im Ordner des Repositorys starten und lab04/ oeffnen.")
DATA     = REPO_ROOT / "data"                           # der Operator / setzt Pfade zusammen
SM       = DATA / "stoermeldungen"                      # train.jsonl, val.jsonl, eval.jsonl, regression.jsonl
MODELL   = DATA / "models" / "SmolLM2-135M-Instruct"     # 134,5 Mio. Parameter, 538 MB in fp32
RESULTS  = Path("results")                              # hierhin schreibt das Notebook Logs und den Adapter
REFDIR   = Path("results_referenz")                     # Referenzlogs und Referenz-Adapter (Fallback)

print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | "
      f"Threads: {torch.get_num_threads()} | CPU-Kerne: {os.cpu_count()}")
print(f"REPO_ROOT: {REPO_ROOT}\n")
check("Hub-Zugriff abgeschaltet (HF_HUB_OFFLINE=1 vor dem Import gesetzt)", huggingface_hub.constants.HF_HUB_OFFLINE,
      "Kernel neu starten und diese Zelle als ERSTE ausfuehren")
# Zeilen je Datei zaehlen: eine .jsonl-Datei enthaelt ein JSON-Objekt pro Zeile
n = {k: sum(1 for _ in open(SM / f"{k}.jsonl", encoding="utf-8")) if (SM / f"{k}.jsonl").exists() else 0
     for k in ("train", "val", "eval", "regression")}
check(f"Daten data/stoermeldungen: {n}", n == {"train": 300, "val": 30, "eval": 50, "regression": 10}, "git pull")
check("SmolLM2-135M-Instruct in data/models/ (config, tokenizer, model.safetensors)",
      all((MODELL / f).exists() for f in ["config.json", "tokenizer.json", "model.safetensors"]),
      "fehlt -> die Download-Zelle davor (oder setup_check.ipynb) mit Internet ausfuehren")
RESULTS.mkdir(exist_ok=True)                            # Ordner results/ anlegen, falls er noch fehlt
if not all(CHECKS.values()):
    raise FileNotFoundError("Setup unvollstaendig (siehe ❌ oben).")
# Der Referenz-Adapter wird nur fuer den Fallback in Aufgabe 4 gebraucht
check("Referenz-Adapter results_referenz/adapter_stoermeldung/ (Fallback fuer Aufgabe 4)",
      (REFDIR / "adapter_stoermeldung" / "adapter_config.json").exists(), "git pull")

### Werkzeuge

`AUSFÜHREN`. Vorgegeben sind Hilfsfunktionen, die in mehreren Aufgaben gebraucht werden:

| Funktion | Was sie tut | gebraucht in |
|---|---|---|
| `lade_jsonl` | liest einen Datensplit (eine Zeile = ein Beispiel mit `meldung`, `ziel`, `art`, `id`) | hier |
| `ziel_json` | macht aus dem Ziel-Dictionary den JSON-Text, den das Modell lernen soll | 2, 3 |
| `chat` | baut die Nachrichtenliste aus Systemprompt, optionalen Beispielen (Few-Shot) und der Meldung | 2, 3, 6 |
| `generiere` | lässt ein Modell **greedy** antworten, 10 Prompts auf einmal | 2, 6, 7 |
| `save_log`, `lade_referenz` | Ergebnisse als JSON nach `results/` schreiben bzw. Referenzlog laden | überall |
| `intervall` | 95-%-Intervall einer Quote (Formel in Aufgabe 6) | 6 |

`bewerte` (Aufgabe 2) und `vorzeichentest` (Aufgabe 6) vervollständigen Sie später selbst.

In [ ]:
FELDER = ["anlage", "symptom", "messwert", "ursache", "massnahme"]          # die fuenf Felder des Ziel-JSON
SYSTEM = ("Extrahiere aus der Stoermeldung die Felder anlage, symptom, messwert, ursache und massnahme. "
          "Antworte nur mit einem JSON-Objekt. Fehlende Angaben sind null.")   # Systemprompt: die Arbeitsanweisung

def lade_jsonl(name):
    """Liest data/stoermeldungen/<name>.jsonl: jede Zeile ist ein JSON-Objekt -> Liste von Dictionaries."""
    with open(SM / f"{name}.jsonl", encoding="utf-8") as f:
        return [json.loads(z) for z in f]

# Alle vier Splits einmal einlesen. Jedes Beispiel ist ein Dictionary, z. B. train[0]["meldung"] oder train[0]["ziel"]
train, val, ev, regr = lade_jsonl("train"), lade_jsonl("val"), lade_jsonl("eval"), lade_jsonl("regression")

def ziel_json(ziel):
    """Kanonische Zielantwort: kompaktes JSON, feste Schluesselreihenfolge, Umlaute im Klartext."""
    return json.dumps({k: ziel[k] for k in FELDER}, ensure_ascii=False)   # ensure_ascii=False: "ä" statt "ä"

def chat(meldung, beispiele=()):
    """Nachrichtenliste: Systemprompt, optional Few-Shot-Beispiele als (user, assistant)-Paare, dann die Meldung."""
    msgs = [{"role": "system", "content": SYSTEM}]
    for b in beispiele:                                 # Few-Shot: jedes Beispiel als Frage (user) und Antwort (assistant)
        msgs += [{"role": "user", "content": b["meldung"]}, {"role": "assistant", "content": ziel_json(b["ziel"])}]
    return msgs + [{"role": "user", "content": meldung}]   # zuletzt die Meldung, die das Modell beantworten soll

def generiere(model, tok, chats, max_new_tokens=96, batch_size=10):
    """Greedy-Dekodierung (do_sample=False) in Batches; Padding links. Rueckgabe: Liste der Antworttexte."""
    tok.padding_side = "left"           # kuerzere Prompts links auffuellen, damit alle Antworten direkt dahinter beginnen
    model.eval(); out = []              # eval(): Dropout aus -> reproduzierbare Ausgaben
    for i in range(0, len(chats), batch_size):          # immer 10 Prompts auf einmal
        # Chat-Template: Nachrichtenliste -> ein Text mit Rollenmarken; add_generation_prompt haengt "<|im_start|>assistant" an
        texte = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in chats[i:i + batch_size]]
        enc = tok(texte, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)  # Text -> Token-IDs
        with torch.inference_mode():                    # nur rechnen, keine Gradienten
            g = model.generate(**enc, do_sample=False, max_new_tokens=max_new_tokens,          # do_sample=False = greedy
                               pad_token_id=tok.pad_token_id, eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"))
        # g enthaelt Prompt + Antwort; wir schneiden den Prompt ab und machen aus den neuen IDs wieder Text
        out += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in out]                     # Leerzeichen/Zeilenumbrueche am Rand entfernen

def save_log(name, **inhalt):
    """Schreibt die uebergebenen Werte (plus Seed, Versionen) als results/<name>.json."""
    log = {"seed": 0, "torch": torch.__version__, "threads": torch.get_num_threads(), **inhalt}
    (RESULTS / f"{name}.json").write_text(json.dumps(log, indent=1, ensure_ascii=False, default=float))
    return log

def lade_referenz(name):
    """Fallback: Referenzlog aus results_referenz/<name>.json."""
    return json.loads((REFDIR / f"{name}.json").read_text(encoding="utf-8"))

def intervall(p, n):
    """95-%-Intervall nach Skript Gl. (SE) als (unten, oben): p ± 1,96 * sqrt(p (1 - p) / n), auf [0, 1] begrenzt."""
    se = math.sqrt(p * (1 - p) / n)                     # Standardfehler einer Quote p auf n Beispielen
    return max(0.0, p - 1.96 * se), min(1.0, p + 1.96 * se)

print(f"geladen: {len(train)} Train, {len(val)} Val, {len(ev)} Eval, {len(regr)} Regression")
print("Beispiel train[0]:", train[0]["meldung"])

## Aufgabe 1 · Das Modell von innen

`PRINZIP` 1.1, 1.2 · `CODE` 1.3 (1 Zeile) · `DENKEN` · Block nach der Folie „Leitbeispiel und Modell“ · Skript Abschn. 1.4 · ≈ 7 Min.

**Lernziel** (Verstehen): den Bauplan eines Decoder-Sprachmodells benennen und seine Parameterzahl aus wenigen Kenngrößen herleiten.

**Intuition.** Ein Decoder wie SmolLM2 ist ein Stapel aus 30 *gleich gebauten* Schichten zwischen einer Embedding-Tabelle
(Token-ID → Vektor mit 576 Zahlen) und einer Ausgabeschicht (Vektor → 49 152 Logits, eines je Token im Vokabular). Jede Schicht
besteht aus Attention (vier Matrizen $\mathbf W_q, \mathbf W_k, \mathbf W_v, \mathbf W_o$) und einem Feed-Forward-Netz (drei Matrizen).
Mehr steckt nicht darin – deshalb lässt sich die Parameterzahl mit dem Taschenrechner nachrechnen.

**Kenngrößen von SmolLM2-135M-Instruct**

| Symbol | Bedeutung | Wert |
|---|---|---|
| $N$ | Zahl der Decoder-Schichten | 30 |
| $d$ | Modellbreite (Länge jedes Token-Vektors) | 576 |
| $d_{ff}$ | Breite des Feed-Forward-Netzes | 1 536 |
| $\lvert\mathcal V\rvert$ | Vokabulargröße | 49 152 |
| $h$, $h_{kv}$ | Query-Köpfe bzw. Key/Value-Köpfe (je 64 Dimensionen) | 9 bzw. 3 |
| $d_{kv} = h_{kv}\cdot 64$ | Breite von Key- und Value-Projektion (*Grouped-Query Attention*) | 192 |

### 1.1 Der Bauplan

`PRINZIP`. `print(basis)` zeigt alle Bausteine. Lesen Sie von oben nach unten: `embed_tokens` → `30 x LlamaDecoderLayer`
(je `self_attn` + `mlp` + zwei Normen) → `norm` → `lm_head`. `Linear(in_features=576, out_features=192)` heißt: Diese Matrix bildet
576 Zahlen auf 192 ab; gespeichert ist sie in der Form 192 × 576.

In [ ]:
abschnitt("Aufgabe 1")
tok = AutoTokenizer.from_pretrained(MODELL)             # Tokenizer: Text <-> Token-IDs (aus demselben Ordner wie das Modell)
torch.manual_seed(0)
# Das vortrainierte Modell laden. float32 = 4 Byte je Zahl -> 134,5 Mio. Parameter * 4 Byte ≈ 538 MB
basis = AutoModelForCausalLM.from_pretrained(MODELL, torch_dtype=torch.float32).to(DEVICE)   # das unveraenderte Modell
print(basis)                                            # zeigt den Aufbau als verschachtelte Liste von Modulen

### 1.2 Eine Schicht im Detail

`PRINZIP`. Die Kenngrößen aus der Konfiguration und die Formen aller Gewichte der ersten Schicht. Achten Sie auf `k_proj` und
`v_proj`: nur 192 statt 576 Zeilen. Das ist Grouped-Query Attention – 3 Key/Value-Köpfe à 64 Dimensionen für 9 Query-Köpfe.

In [ ]:
cfg = basis.config                                      # die Kenngroessen stehen in der Konfiguration des Modells
print("Schichten N          :", cfg.num_hidden_layers)
print("Modellbreite d_model :", cfg.hidden_size)
print("FFN-Breite d_ff      :", cfg.intermediate_size)
print("Query-/KV-Koepfe     :", cfg.num_attention_heads, "/", cfg.num_key_value_heads)
print("Vokabular |V|        :", cfg.vocab_size)
print()
schicht = basis.model.layers[0]                         # die erste der 30 Decoder-Schichten
# named_parameters() liefert jedes Gewicht mit Namen; p.shape = Form, p.numel() = Anzahl Zahlen (Zeilen * Spalten)
for name, p in schicht.named_parameters():
    print(f"{name:34} Form {str(tuple(p.shape)):12} {p.numel():>9,} Zahlen")

### 1.3 Parameter von Hand zählen

`CODE` (1 Zeile). Eine Matrix der Form $m\times n$ hat $m\cdot n$ Einträge. Die Linearschichten von SmolLM2 haben keinen Bias. Damit:

$$
\begin{aligned}
\text{Embedding} &= \lvert\mathcal V\rvert\cdot d \\
\text{Attention je Schicht} &= \underbrace{d\cdot d}_{\mathbf W_q} + \underbrace{d\cdot d_{kv}}_{\mathbf W_k} + \underbrace{d\cdot d_{kv}}_{\mathbf W_v} + \underbrace{d\cdot d}_{\mathbf W_o} \\
\text{FFN je Schicht} &= 3\cdot d\cdot d_{ff} \quad(\texttt{gate\_proj},\ \texttt{up\_proj},\ \texttt{down\_proj}) \\
\text{Normen je Schicht} &= 2\cdot d \quad(\text{zwei RMSNorm-Vektoren}) \\
\text{gesamt} &= \text{Embedding} + N\cdot(\text{Attention} + \text{FFN} + \text{Normen}) + d \quad(\text{finale Norm})
\end{aligned}
$$

Embedding und Ausgabeschicht teilen sich dieselbe Matrix (*weight tying*) – sie wird nur einmal gezählt. Ergänzen Sie `je_schicht` aus
den drei Teilsummen. **Erwartung:** 3 540 096 Parameter je Schicht, 134 515 008 insgesamt – exakt die Zahl, die PyTorch zählt.

In [ ]:
d, d_ff, V, N = 576, 1536, 49152, 30                    # Kenngroessen aus der Tabelle oben
d_kv = 3 * 64                                           # 3 Key/Value-Koepfe a 64 Dimensionen = 192

embedding = V * d                                       # eine Zeile mit d Zahlen je Token im Vokabular
attention = d * d + d * d_kv + d * d_kv + d * d         # W_q, W_k, W_v, W_o
ffn       = 3 * d * d_ff                                # gate_proj, up_proj, down_proj
normen    = 2 * d                                       # zwei RMSNorm-Vektoren
# TODO 1.3: Parameter je Schicht = Summe aus attention, ffn und normen
raise NotImplementedError("TODO 1.3")
gesamt = embedding + N * je_schicht + d                 # + finale Norm; lm_head teilt sich die Gewichte mit embedding

print(f"Embedding  : {embedding:>12,}")                  # :>12, = rechtsbuendig, 12 Zeichen breit, mit Tausendertrennzeichen
print(f"je Schicht : {je_schicht:>12,}   (Attention {attention:,} + FFN {ffn:,} + Normen {normen:,})")
print(f"gesamt     : {gesamt:>12,}   von Hand")
print(f"gesamt     : {sum(p.numel() for p in basis.parameters()):>12,}   gezaehlt von PyTorch")
# data_ptr() ist die Speicheradresse: gleiche Adresse = dieselbe Matrix (weight tying)
print("lm_head ist dieselbe Matrix wie embed_tokens:", basis.lm_head.weight.data_ptr() == basis.model.embed_tokens.weight.data_ptr())
check("1.3 Parameter von Hand = PyTorch (134 515 008)", lambda: gesamt == sum(p.numel() for p in basis.parameters()))

### Interpretation Aufgabe 1

`DENKEN`

1. Welcher Baustein hat die meisten Parameter: Embedding, Attention oder Feed-Forward-Netz (alle 30 Schichten zusammen)?
2. Ohne weight tying käme die Ausgabeschicht als eigene Matrix dazu. Um wie viel Prozent wüchse das Modell?

**Ihre Antwort:**

## Aufgabe 2 · Der Decoder von Hand und die Prompting-Baseline

`PRINZIP` 2.1–2.4 · `CODE` 2.2–2.5 (je 1 Zeile) · `AUSFÜHREN` 2.6 · `DENKEN` · Skript Abschn. 2.1–2.3 · ≈ 24 Min. + ≈ 1–3 Min. Rechenzeit

**Zwei Blöcke:** 2.1–2.2 nach der Folie „Beispiel: Perplexität von Hand“ (8 Min.), 2.3–2.6 nach „Zwischenstand: Decoder“ (16 Min.).

**Lernziel** (Verstehen/Anwenden): nachvollziehen, wie ein Decoder aus Logits Wahrscheinlichkeiten, aus Wahrscheinlichkeiten einen
Verlust und aus einer Schleife über „nächstes Token“ einen Text macht – und die Prompting-Baseline messen.

**Intuition.** Das Modell liefert an *jeder* Position einen Vektor mit 49 152 Logits (unnormierte Bewertungen, eine je Token im
Vokabular). Alles Weitere ist einfache Rechnung:
Softmax → Wahrscheinlichkeiten · $-\ln p$ des tatsächlichen Tokens → Verlust · $e^{\text{mittlerer Verlust}}$ → Perplexität ·
Logits durch $T$ teilen → Temperatur · immer das größte nehmen und anhängen → greedy Generierung.

### 2.1 Das nächste Token

`PRINZIP`. Dieselbe Eingabe wie in der Live-Demo mit GPT-2 im Browser, jetzt mit SmolLM2. Für die Fortsetzung zählt nur die letzte
Position. Aus den Logits $a_1,\dots,a_{|\mathcal V|}$ macht die **Softmax** Wahrscheinlichkeiten, die positiv sind und sich zu 1 addieren:

$$p_i = \frac{e^{a_i}}{\sum_{j=1}^{|\mathcal V|} e^{a_j}}.$$

In [ ]:
abschnitt("Aufgabe 2")
text = "The machine stopped because the"
ids = tok(text, return_tensors="pt")["input_ids"]       # Text -> Tensor der Form (1, Anzahl Token) mit Token-IDs
print("Token:", tok.convert_ids_to_tokens(ids[0]))      # die Token als Text ("Ġ" steht fuer ein Leerzeichen davor)

with torch.no_grad():                                   # nur vorwaerts rechnen, keine Gradienten noetig
    logits = basis(ids).logits                          # Form (1, Anzahl Token, 49152): eine Vorhersage je Position
print("Form der Logits:", tuple(logits.shape))

letzte = logits[0, -1]                                  # Satz 0, letzte Position: Vorhersage fuer das NAECHSTE Token
p = torch.softmax(letzte, dim=-1)                       # Softmax-Formel oben -> 49 152 Wahrscheinlichkeiten
print(f"Summe aller Wahrscheinlichkeiten: {float(p.sum()):.4f}\n")
top = torch.topk(p, 5)                                  # die 5 groessten Werte und ihre Positionen (= Token-IDs)
for wahrsch, i in zip(top.values, top.indices):         # zip laeuft parallel durch beide Listen
    print(f"{tok.decode(int(i))!r:12} p = {float(wahrsch):.3f}")   # decode: Token-ID -> Text

### 2.2 Satzwahrscheinlichkeit, Verlust und Perplexität

`CODE` (1 Zeile) + `PRINZIP`. Ein Sprachmodell zerlegt die Wahrscheinlichkeit eines Satzes $y_1,\dots,y_n$ in Faktoren – jedes Token
bedingt auf alle vorherigen (Produktregel). Der Verlust je Token ist der negative Logarithmus, die **Perplexität** (Skript Gl. PPL) das
Exponential des mittleren Verlusts:

$$
P(\text{Satz}) = \prod_{t=1}^{n} p_t,\qquad
\ell_t = -\ln p_t,\qquad
\bar L = \frac1n\sum_{t=1}^{n}\ell_t,\qquad
\mathrm{PPL} = e^{\bar L} = \Bigl(\prod_{t=1}^{n} p_t\Bigr)^{-1/n}
\quad\text{mit } p_t = p(y_t\mid y_{<t}).
$$

Die Perplexität ist die effektive Zahl gleich wahrscheinlicher Kandidaten, zwischen denen das Modell im Mittel je Token schwankt
(perfektes Modell: PPL = 1). **Rechenbeispiel** (Folie „Beispiel: Perplexität von Hand“): Die vier Token einer Antwort haben
$p = 0{,}5;\ 0{,}25;\ 0{,}8;\ 0{,}1$. Ergänzen Sie die Zeile für `ppl`. **Erwartung:** PPL = 3,16.

In [ ]:
p_token = [0.5, 0.25, 0.8, 0.1]                         # Wahrscheinlichkeiten der vier Token (Rechenbeispiel)
p_satz = 1.0
for p_t in p_token:
    p_satz = p_satz * p_t                               # Produktregel: alle Faktoren multiplizieren
verluste = [-math.log(p_t) for p_t in p_token]          # Verlust je Token: -ln p (math.log = natuerlicher Logarithmus)
mittel = sum(verluste) / len(verluste)                  # mittlerer Verlust
# TODO 2.2: Perplexitaet = e hoch mittlerer Verlust (math.exp)
raise NotImplementedError("TODO 2.2")
print(f"p(Satz) = {p_satz:.3f} | Verluste {[round(v, 3) for v in verluste]} | Mittel {mittel:.3f} | PPL {ppl:.2f}")
check("2.2 Perplexitaet des Folienbeispiels = 3,16", lambda: abs(ppl - 3.162) < 1e-3)

`PRINZIP`. Jetzt liefert das Modell die Wahrscheinlichkeiten $p_t$ für eine echte Störmeldung. **Wichtig:** Die Vorhersage an Position
$t-1$ ist die Verteilung für Token $t$ – deshalb läuft die Schleife ab $t = 1$, und das erste Token bekommt keinen Verlust.
Übergibt man `labels=ids`, macht die Bibliothek diese Verschiebung intern; das Ergebnis muss gleich sein.

In [ ]:
satz = "Presse 3 steht seit 06:40, der Hydraulikdruck fällt ab."
ids = tok(satz)["input_ids"]                            # hier als normale Python-Liste von Token-IDs
with torch.no_grad():
    logits = basis(torch.tensor([ids])).logits[0]       # [ids]: ein Batch mit einem Satz; [0]: diesen Satz herausnehmen
                                                        # -> Form (Anzahl Token, 49152)
verluste = []
for t in range(1, len(ids)):                            # t = 1, 2, ..., letzte Position (Token 0 hat keinen Vorgaenger)
    p_vert = torch.softmax(logits[t - 1], dim=-1)       # Vorhersage an Position t-1 ...
    p_t = float(p_vert[ids[t]])                         # ... davon die Wahrscheinlichkeit des tatsaechlichen Tokens t
    verluste.append(-math.log(p_t))                     # Verlust dieses Tokens an die Liste anhaengen
    print(f"{tok.decode(ids[t])!r:10} p = {p_t:.3f}   Verlust = {-math.log(p_t):5.2f}")

ppl_hand = math.exp(sum(verluste) / len(verluste))      # Perplexitaet = e hoch mittlerer Verlust
with torch.no_grad():
    loss_bib = basis(torch.tensor([ids]), labels=torch.tensor([ids])).loss   # Verschiebung um 1 passiert intern
print(f"\nPerplexitaet von Hand: {ppl_hand:.2f} | aus der Bibliothek: {math.exp(float(loss_bib)):.2f}")

### 2.3 Temperatur und Top-p von Hand

`CODE` (1 Zeile) + `PRINZIP`. Zwei Stellschrauben beim *Sampling* (Ziehen aus der Verteilung statt immer das Maximum zu nehmen):

**Temperatur** (Skript Gl. Temperatur): Die Logits werden vor der Softmax durch $T$ geteilt.

$$p_i(T) = \frac{e^{a_i/T}}{\sum_j e^{a_j/T}}\qquad
T<1:\ \text{spitzer (nahe greedy)},\quad T=1:\ \text{gelernte Verteilung},\quad T>1:\ \text{flacher (zufälliger)}.$$

**Top-p** (*Nucleus Sampling*): nur die kleinste Menge der wahrscheinlichsten Token behalten, deren Summe die Schwelle $p$ erreicht –
also absteigend sortieren, aufsummieren, bis die Schwelle erreicht ist, den Rest auf 0 setzen und durch die Summe teilen (renormieren).

Rechenbeispiel mit den Logits $\mathbf a = (2;\ 1;\ 0{,}5;\ -1)$ (Folie „Beispiel 2.2“). Ergänzen Sie in `softmax_T` die Zeile für `z`;
`top_p` ist als einfache Schleife vorgegeben. **Erwartung:** $T=1$: (0,609; 0,224; 0,136; 0,030) · Top-p 0,8: (0,731; 0,269; 0; 0).

In [ ]:
a = np.array([2.0, 1.0, 0.5, -1.0])                     # Logits aus dem Rechenbeispiel

def softmax_T(a, T):
    """Softmax mit Temperatur T: e^(a_i / T), dann durch die Summe teilen."""
    # TODO 2.3: z = e hoch (Logits / T), elementweise mit np.exp
    raise NotImplementedError("TODO 2.3")
    return z / z.sum()                                  # normieren: Summe 1

def top_p(p, schwelle):
    """Behaelt die wahrscheinlichsten Token, bis ihre Summe die Schwelle erreicht; Rest = 0; renormiert."""
    reihenfolge = np.argsort(-p)                        # Indizes, groesste Wahrscheinlichkeit zuerst
    behalten, summe = [], 0.0
    for i in reihenfolge:
        behalten.append(i)                              # Token i kommt in den "Topf"
        summe += p[i]                                   # kumulierte Wahrscheinlichkeit
        if summe >= schwelle:                           # genug Wahrscheinlichkeitsmasse beisammen
            break                                       # -> Schleife verlassen
    q = np.zeros_like(p)                                # Vektor aus Nullen in derselben Form wie p
    q[behalten] = p[behalten]                           # nur die behaltenen Token bekommen ihre Wahrscheinlichkeit zurueck
    return q / q.sum()                                  # renormieren

check("2.3 Softmax bei T = 1 wie auf der Folie", lambda: np.allclose(softmax_T(a, 1.0), [0.609, 0.224, 0.136, 0.030], atol=1e-3))
for T in (0.5, 1.0, 2.0):
    print(f"T = {T}: {np.round(softmax_T(a, T), 3)}")
print(f"Top-p 0,8 bei T = 1: {np.round(top_p(softmax_T(a, 1.0), 0.8), 3)}")

`PRINZIP`. Dieselben zwei Funktionen auf der echten Verteilung aus 2.1 (49 152 Kandidaten): Wie viele Token bleiben bei Top-p 0,9 im
Topf? Und was kommt heraus, wenn man aus dem Topf zieht statt das Maximum zu nehmen?

In [ ]:
logits_2_1 = letzte.numpy().astype(np.float64)          # die Logits aus 2.1 als NumPy-Vektor (64-Bit fuer genaue Summen)
for T in (0.5, 1.0, 2.0):
    q = top_p(softmax_T(logits_2_1, T), 0.9)
    print(f"T = {T}: {int((q > 0).sum()):>6} von {len(q)} Token bleiben im Topf")   # (q > 0).sum() zaehlt die Nicht-Nullen

rng = np.random.default_rng(0)                          # Zufallsgenerator mit festem Startwert
q = top_p(softmax_T(logits_2_1, 1.0), 0.9)
# rng.choice zieht eine Token-ID; jede ID mit der Wahrscheinlichkeit aus q. Das 8-mal wiederholen:
gezogen = [tok.decode(int(rng.choice(len(q), p=q))) for _ in range(8)]
print("\n8-mal gezogen (T = 1, Top-p 0,9):", gezogen)
print("greedy (immer das Maximum)      :", tok.decode(int(np.argmax(logits_2_1))))

### 2.4 Greedy-Generierung als Schleife

`CODE` (1 Zeile) + `PRINZIP`. Greedy-Dekodierung nimmt in jedem Schritt das Token mit dem größten Logit (die Softmax ändert die
Reihenfolge nicht und ist dafür unnötig) und hängt es an:

$$y_{t+1} = \arg\max_{i}\ a_i^{(t)}\qquad(a^{(t)} = \text{Logits an der letzten Position } t).$$

`model.generate` ist im Kern diese Schleife: Logits für die letzte Position, das größte nehmen, anhängen, wiederholen – bis das
Ende-Token `<|im_end|>` kommt. Ohne Zwischenspeicher (KV-Cache) rechnet die Schleife in jedem Schritt die ganze Folge neu; das ist
langsamer, das Ergebnis aber dasselbe. Sehen Sie sich den Prompt an: Das Chat-Template hat einen Systemprompt ergänzt, den niemand
geschrieben hat (mehr dazu in Aufgabe 3). **Erwartung:** eigene Schleife = `model.generate` (Prüfung ✅).

In [ ]:
frage = [{"role": "user", "content": "Was macht ein Drehmomentschlüssel? Antworte in einem Satz."}]
prompt = tok.apply_chat_template(frage, tokenize=False, add_generation_prompt=True)   # Nachricht -> Text mit Rollenmarken
print(prompt)
ids = tok(prompt, add_special_tokens=False)["input_ids"]   # Prompt als Liste von Token-IDs
ENDE = tok.convert_tokens_to_ids("<|im_end|>")          # ID des Ende-Tokens: hier hoert die Antwort auf

neu = []                                                # die neu erzeugten Token-IDs
for schritt in range(40):                               # hoechstens 40 neue Token
    with torch.no_grad():
        logits = basis(torch.tensor([ids + neu])).logits[0, -1]   # ganze Folge neu rechnen; nur die letzte Position zaehlt
    # TODO 2.4: greedy: Index des groessten Logits als int (torch.argmax)
    raise NotImplementedError("TODO 2.4")
    if naechstes == ENDE:                               # Ende-Token -> fertig
        break
    neu.append(naechstes)                               # sonst anhaengen und im naechsten Schritt mitverarbeiten
    if schritt < 5:
        print(f"Schritt {schritt}: {tok.decode(naechstes)!r}")

print("\nSchleife :", tok.decode(neu))
mit_generate = generiere(basis, tok, [frage], max_new_tokens=40)[0]   # dasselbe mit der Bibliothek (greedy)
print("generate :", mit_generate)
check("2.4 eigene Schleife = model.generate (greedy)", lambda: tok.decode(neu).strip() == mit_generate)

### 2.5 Prompting-Baseline: Bewertungsfunktion

`CODE` (1 Zeile) · Skript Abschn. 2.3 (Prompting-Baseline), 8.1 (Listing regelbasierte Evaluation), Eval-Karte

**Lernziel** (Anwenden/Analysieren): Die Prompting-Baseline ist die billigste Anpassung – und der Maßstab, gegen den sich jedes
Training rechtfertigen muss. Sie setzen eine regelbasierte, gestufte Bewertung um und messen damit die Baseline.

Die Stufen (Skript Abschn. 8.1) bauen aufeinander auf:

| Stufe | bestanden, wenn … |
|---|---|
| **(a) Format** (`format_ok`) | `json.loads(ausgabe)` gelingt und liefert ein JSON-Objekt (in Python: ein Dictionary) |
| **(b) Schema** (`schema_ok`) | (a) bestanden **und** alle fünf Felder aus `FELDER` sind als Schlüssel vorhanden (zusätzliche Felder sind erlaubt) |
| **(c) exakt** (`anlage_ok`, `messwert_ok`) | (b) bestanden und der Wert stimmt nach Normalisierung (Kleinschreibung, Leerraum vereinheitlicht) mit dem Ziel überein – diese beiden Felder werden maschinell weiterverarbeitet |
| **komplett** | (b) und beide Felder aus (c) exakt |
| **streng** | alle fünf Felder exakt – unterschätzt die Qualität, weil `symptom`, `ursache`, `massnahme` auch anders richtig formuliert sein können (dafür bräuchte man einen Judge, Optional O1) |

Ergänzen Sie in `bewerte` die Zeile für Stufe (b). In Python prüft `set(A) <= set(B)`, ob jedes Element von `A` auch in `B` vorkommt
(Teilmenge); `set(pred)` ist bei einem Dictionary die Menge seiner Schlüssel.

In [ ]:
EXAKT = ["anlage", "messwert"]                          # Felder, die exakt stimmen muessen (Stufe c)

def norm(s):
    """Normalisierung fuer den Vergleich: Kleinschreibung, mehrfache Leerzeichen zu einem; None bleibt None."""
    return None if s is None else " ".join(str(s).lower().split())

def bewerte(ausgabe, ziel):
    """Gestufte, regelbasierte Bewertung einer Modellausgabe gegen das Ziel (Skript Listing regelbasierte Evaluation)."""
    erg = {"format_ok": False, "schema_ok": False, "anlage_ok": False, "messwert_ok": False,
           "komplett": False, "streng": False}           # zunaechst alles "nicht bestanden"
    try:
        pred = json.loads(ausgabe)                      # Text -> Python-Objekt; scheitert bei ungueltigem JSON
    except json.JSONDecodeError:
        return erg                                      # Stufe (a) nicht bestanden
    erg["format_ok"] = isinstance(pred, dict)           # nur ein JSON-Objekt {...} zaehlt, keine Liste oder Zahl
    # TODO 2.5: schema_ok: format_ok UND alle Felder aus FELDER sind Schluessel von pred (Mengenvergleich mit <=)
    raise NotImplementedError("TODO 2.5")
    if erg["schema_ok"]:
        for f in EXAKT:                                 # Stufe (c): anlage und messwert exakt?
            erg[f"{f}_ok"] = norm(pred[f]) == norm(ziel[f])
        erg["komplett"] = erg["anlage_ok"] and erg["messwert_ok"]
        erg["streng"] = all(norm(pred[f]) == norm(ziel[f]) for f in FELDER)   # all(...): wahr, wenn alle fuenf stimmen
    return erg

def auswerten(ausgaben, daten, name):
    """Bewertet alle Ausgaben, gibt Quoten je Stufe zurueck und schreibt das Log."""
    bew = [bewerte(a, b["ziel"]) for a, b in zip(ausgaben, daten)]          # eine Bewertung je Meldung
    quoten = {k: float(np.mean([e[k] for e in bew])) for k in bew[0]}       # Mittelwert von True/False = Anteil bestanden
    save_log(name, quoten=quoten, n=len(daten), einzeln=bew, ausgaben=ausgaben, ids=[b["id"] for b in daten])
    return quoten, bew

z = ev[0]["ziel"]                                       # ein Ziel aus dem Eval-Set fuer die Pruefungen
check("2.5 gueltiges, vollstaendiges JSON -> schema_ok", lambda: bewerte(ziel_json(z), z)["schema_ok"] and bewerte(ziel_json(z), z)["streng"])
check("2.5 fehlendes Feld -> schema_ok False", lambda: not bewerte(json.dumps({"anlage": z["anlage"]}), z)["schema_ok"])
check("2.5 Fliesstext statt JSON -> format_ok False", lambda: not bewerte("Die Anlage ist Presse 6.", z)["format_ok"])

### 2.6 Zero-Shot und Few-Shot

`AUSFÜHREN`. Dasselbe Modell, dieselbe Dekodierung (greedy, `max_new_tokens=96`) auf den 50 Eval-Meldungen – einmal nur mit
Systemprompt (**zero-shot**), einmal mit drei gelösten Beispielen aus dem Trainingsset vor der eigentlichen Meldung (**few-shot**).
**Erwartung** (Lösungslauf): zero-shot 0 % gültiges JSON – das Modell wiederholt oder übersetzt die Meldung; few-shot 94 % Format,
78 % Schema, 48 % `anlage` exakt, 36 % `messwert` exakt, 22 % komplett.

In [ ]:
FEWSHOT = [train[i] for i in (1, 2, 3)]                 # feste Auswahl: typisch, ohne Messwert, mit Abkuerzungen
t0 = time.perf_counter()                                # Startzeit fuer die Zeitmessung
aus_zero = generiere(basis, tok, [chat(b["meldung"]) for b in ev])            # zero-shot: nur Systemprompt + Meldung
aus_few = generiere(basis, tok, [chat(b["meldung"], FEWSHOT) for b in ev])    # few-shot: drei Beispiele davor
print(f"Generierung 2 x 50 Meldungen: {time.perf_counter() - t0:.0f} s")
q_zero, bew_zero = auswerten(aus_zero, ev, "a2_zero_shot")                    # Quoten je Stufe + Einzelbewertungen
q_few, bew_few = auswerten(aus_few, ev, "a2_few_shot")
display(pd.DataFrame({"zero-shot": q_zero, "few-shot (3)": q_few}).T.style.format("{:.0%}"))   # Tabelle in Prozent
for i in (0, 41):                                       # zwei Beispiele zum Lesen
    print(f"\n[{ev[i]['id']}] {ev[i]['meldung']}\n  zero-shot: {aus_zero[i][:160]!r}\n  few-shot : {aus_few[i][:160]!r}")

### Interpretation Aufgabe 2

`DENKEN`

1. In 2.1 hat das wahrscheinlichste Token nur rund 4 %. Wie viele Kandidaten bleiben bei $T=1$ und Top-p 0,9 – und warum ist
   Sampling deshalb für eine JSON-Extraktion ungeeignet?
2. Die Störmeldung in 2.2 hat eine Perplexität um 80, das Rechenbeispiel 3,16. Ist SmolLM2 deshalb für unsere Aufgabe „schlecht“?
   Wofür taugt Perplexität, wofür nicht?
3. Die Schleife in 2.4 und `generate` liefern denselben Text. Was spart der KV-Cache – und warum ändert er das Ergebnis nicht?
   (Denken Sie an die kausale Maske: Position $t$ sieht nur die Positionen davor.)
4. Welche Stufe scheitert ohne Anpassung am häufigsten – und was sagt das über die Grenzen des Promptings (Skript Abschn. 2.3)?
5. Few-Shot kostet bei jeder Anfrage zusätzliche Token. Schätzen Sie mit `len(tok(...)["input_ids"])` grob, wie viele – und
   vergleichen Sie mit dem Gewinn.
6. Warum ist „streng“ (alle Felder exakt) kein fairer Maßstab für `symptom` oder `ursache`?

**Ihre Antwort:**

## Aufgabe 3 · Instruction-Tuning: Daten, Chat-Template und Loss-Maske

`AUSFÜHREN` 3.0 · `PRINZIP` 3.1 · `CODE` 3.2 (1 Zeile), 3.3 (2 Zeilen) · `DENKEN` · Block nach der Folie „Zwischenstand:
Instruction-Tuning“ · Skript Bsp. 1.1, Abschn. 4.1 (Listing ChatML), 4.3 · ≈ 10 Min.

**Lernziel** (Verstehen): erklären, wie ein Trainingsbeispiel für Instruction-Tuning aufgebaut ist und warum der Verlust nur auf
den Antwort-Token berechnet wird.

**Intuition.** Das Modell sieht beim Training die ganze Folge – Systemprompt, Meldung, Antwort –, soll aber nur lernen, die
*Antwort* zu schreiben. Alle anderen Positionen bekommen das Label $-100$; `CrossEntropyLoss` überspringt sie. Das Ende-Token
`<|im_end|>` gehört zur Antwort: Nur so lernt das Modell, nach dem JSON aufzuhören.

### 3.0 Daten und Splits

`AUSFÜHREN`. Drei Beispiele lesen und prüfen, dass Training und Eval aus verschiedenen Anlagen stammen (*Trennung nach Herkunft*:
Sonst misst das Eval-Set nur, ob das Modell bekannte Anlagen wiedererkennt).

In [ ]:
abschnitt("Aufgabe 3")
print(f"Train {len(train)} | Val {len(val)} | Eval {len(ev)} | Regression {len(regr)}")
print("Eval nach Art:", dict(Counter(b["art"] for b in ev)))   # Counter zaehlt: wie viele typisch / fehlend / abk
anl = lambda d: {b["ziel"]["anlage"].split()[-1] for b in d}   # Menge der Anlagen-Nummern (letztes Wort von "Presse 3")
print("Anlagen-Nummern Train:", sorted(anl(train)), "| Eval:", sorted(anl(ev)), " -> Trennung nach Herkunft")
for b in [train[0], ev[0], ev[45]]:
    print(f"\n[{b['id']}, {b['art']}] {b['meldung']}\n  -> {ziel_json(b['ziel'])}")

### 3.1 Das Chat-Template als Text

`PRINZIP`. Für das Modell ist ein Trainingsbeispiel *eine* Zeichenkette. Das Chat-Template (Format *ChatML*) setzt Rollenmarken um jede
Nachricht:

```
<|im_start|>system
Extrahiere aus der Stoermeldung ...<|im_end|>
<|im_start|>user
Presse 3 steht seit 06:40 ...<|im_end|>
<|im_start|>assistant
{"anlage": "Presse 3", ...}<|im_end|>
```

Genau dieses Format muss auch bei der Inferenz verwendet werden – `generiere` tut das mit `add_generation_prompt=True`, das den Beginn
der Assistentenrolle (`<|im_start|>assistant`) anhängt; das Modell schreibt dann die Antwort.

In [ ]:
b = train[0]
# Nachrichtenliste: System + Meldung (aus chat) und die Zielantwort als Nachricht des Assistenten
nachrichten = chat(b["meldung"]) + [{"role": "assistant", "content": ziel_json(b["ziel"])}]
text = tok.apply_chat_template(nachrichten, tokenize=False)   # tokenize=False: als Text zurueckgeben, nicht als IDs
print(text)
print("Laenge:", len(tok(text, add_special_tokens=False)["input_ids"]), "Token")

### 3.2 Loss-Maske

`CODE` (1 Zeile). `kodiere` baut aus einem Beispiel zwei gleich lange Listen: `input_ids` (die ganze Folge) und `labels` (die Lernziele).
Der Prompt (Systemprompt + Meldung + Beginn der Assistentenrolle, $P$ Token) kommt aus `apply_chat_template`, die Antwort ist das
Ziel-JSON plus `<|im_end|>` ($A$ Token). Die Labels sind

$$
\text{labels} = (\underbrace{-100,\ \dots,\ -100}_{P\text{ mal: Prompt}},\ \underbrace{y_{P+1},\ \dots,\ y_{P+A}}_{\text{Antwort-IDs}}),
$$

anschließend wie `input_ids` auf höchstens `max_len` Einträge gekürzt. **Nicht** selbst um eine Position verschieben – das macht das
Modell intern. In Python: `[-100] * 3` ergibt `[-100, -100, -100]`, `liste1 + liste2` hängt zwei Listen aneinander.
**Ergänzen Sie die Zeile für `labels`.** **Erwartung:** Bei `train[0]` sind 100 Positionen maskiert und 58 Positionen Lernziel; das
letzte Lernziel-Token ist `<|im_end|>`.

In [ ]:
END = "<|im_end|>"

def kodiere(b, max_len=256):
    """Ein Beispiel -> {"input_ids": ganze Folge, "labels": -100 auf dem Prompt, Antwort-IDs auf der Antwort}."""
    prompt = tok.apply_chat_template(chat(b["meldung"]), tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]                    # Prompt-Token (P Stueck)
    a_ids = tok(ziel_json(b["ziel"]) + END, add_special_tokens=False)["input_ids"]  # Antwort-Token inkl. Ende (A Stueck)
    ids = (p_ids + a_ids)[:max_len]                                              # aneinanderhaengen, auf max_len kuerzen
    # TODO 3.2: labels: -100 fuer jede Prompt-Position, danach die Antwort-IDs a_ids (Skript Listing Training); auf max_len kuerzen
    raise NotImplementedError("TODO 3.2")
    return {"input_ids": ids, "labels": labels}

x = kodiere(train[0])
n_maske = sum(l == -100 for l in x["labels"]); n_ziel = len(x["labels"]) - n_maske   # True zaehlt beim Summieren als 1
print(f"Laenge {len(x['input_ids'])} Token: {n_maske} maskiert (Kontext), {n_ziel} Lernziel")
print("Kontext (Ende):", repr(tok.decode(x["input_ids"][n_maske - 12:n_maske])))    # die letzten 12 Prompt-Token
print("Lernziel      :", repr(tok.decode([t for t, l in zip(x["input_ids"], x["labels"]) if l != -100])))

In [ ]:
# Pruefzelle 3.2
check("3.2 labels gleich lang wie input_ids", lambda: len(kodiere(train[0])["labels"]) == len(kodiere(train[0])["input_ids"]))
check("3.2 Prompt-Positionen sind -100, Antwort-Positionen nicht",
      lambda: kodiere(train[0])["labels"][0] == -100 and kodiere(train[0])["labels"][-1] != -100)
check("3.2 letztes Lernziel-Token ist <|im_end|> (Modell lernt aufzuhoeren)",
      lambda: kodiere(train[0])["labels"][-1] == tok.convert_tokens_to_ids(END))
laengen = [len(kodiere(b)["input_ids"]) for b in train]   # Laenge jedes Trainingsbeispiels in Token
print(f"Sequenzlaengen Train: Median {int(np.median(laengen))}, Maximum {max(laengen)} Token (Grenze 256)")

### 3.3 Der Verlust von Hand – mit und ohne Maske

`CODE` (2 Zeilen) + `PRINZIP`. Dieselbe Schleife wie in 2.2, jetzt mit Maske. Sei $M$ die Menge der Positionen, deren Label nicht
$-100$ ist (die Antwort). Der Verlust ist der Mittelwert nur über diese Positionen:

$$L_{\text{Maske}} = \frac{1}{|M|}\sum_{t\in M} -\ln p(y_t\mid y_{<t}),\qquad
L_{\text{ohne}} = \frac{1}{n-1}\sum_{t=2}^{n} -\ln p(y_t\mid y_{<t}).$$

Genau das macht `CrossEntropyLoss(ignore_index=-100)` in der Bibliothek. In der Schleife steht der Verlust an Position `t` für die
Vorhersage des Tokens `t + 1` – also entscheidet `labels[t + 1]`, ob er zählt. Ergänzen Sie die zwei Zeilen, die maskierte Positionen
mit `continue` überspringen (`continue` springt sofort zum nächsten Schleifendurchlauf). **Erwartung:** von Hand = Bibliothek
(≈ 2,03 auf `train[0]`); ohne Maske deutlich höher (≈ 3,5).

In [ ]:
x = kodiere(train[0])
ids, labels = x["input_ids"], x["labels"]
with torch.no_grad():
    logits = basis(torch.tensor([ids])).logits[0]       # (Laenge, 49152): eine Vorhersage je Position

summe_maske, n_maske, summe_alle, n_alle = 0.0, 0, 0.0, 0   # Summen und Zaehler fuer beide Mittelwerte
for t in range(len(ids) - 1):                           # die letzte Position sagt nichts mehr vorher
    verlust = -math.log(float(torch.softmax(logits[t], dim=-1)[ids[t + 1]]))   # Vorhersage an t fuer Token t+1
    summe_alle += verlust                               # ohne Maske: jede Position zaehlt
    n_alle += 1
    # TODO 3.3: Kontext ueberspringen: ist das Label des vorhergesagten Tokens (Position t+1) gleich -100, dann continue
    raise NotImplementedError("TODO 3.3")
    summe_maske += verlust                              # mit Maske: nur Antwort-Positionen
    n_maske += 1

with torch.no_grad():
    loss_bib = basis(torch.tensor([ids]), labels=torch.tensor([labels])).loss   # die Bibliothek rechnet dasselbe
print(f"Verlust mit Maske,  von Hand  : {summe_maske / n_maske:.4f}  ueber {n_maske} Antwort-Token")
print(f"Verlust mit Maske,  Bibliothek: {float(loss_bib):.4f}")
print(f"Verlust ohne Maske, von Hand  : {summe_alle / n_alle:.4f}  ueber {n_alle} Token")
check("3.3 Verlust von Hand = Bibliothek", lambda: abs(summe_maske / n_maske - float(loss_bib)) < 1e-3)

### Interpretation Aufgabe 3

`DENKEN`

1. Was würde das Modell lernen, wenn die Prompt-Positionen **nicht** maskiert wären? Denken Sie an den Systemprompt, der in allen
   300 Beispielen identisch ist (Skript Abschn. 4.1).
2. Die längste Trainingsfolge ist viel kürzer als 256 Token (siehe Ausgabe in 3.2). Was bedeutet *dynamisches Padding* (jeder Batch
   wird nur bis zu seiner längsten Folge aufgefüllt) gegenüber einer festen Sequenzlänge von 256 für die Rechenzeit?
3. In 3.3 ist der Verlust ohne Maske deutlich höher als mit Maske. Welche Token treiben ihn nach oben? Und was würde
   nach einigen Trainingsschritten *ohne* Maske mit dem Systemprompt-Anteil passieren?

**Ihre Antwort:**

## Aufgabe 4 · LoRA konfigurieren und Training starten

`CODE` 4.1 (1 Zeile) · `AUSFÜHREN` 4.2 · `DENKEN` · Block nach der Folie „Demo: LoRA-Training starten“ · Skript Listing PEFT,
Listing Training · ≈ 3 Min. + **≈ 4–7 Min. Rechenzeit, die im Hintergrund der Vorlesung läuft**

**Lernziel** (Anwenden): ein Basismodell mit PEFT um einen LoRA-Adapter ergänzen und das Training starten.

**Worum es geht** (Details in Aufgabe 5): LoRA lässt jede vortrainierte Gewichtsmatrix $\mathbf W_0$ unverändert und lernt daneben
zwei schmale Matrizen $\mathbf A$ ($r\times k$) und $\mathbf B$ ($d\times r$):

$$\mathbf h = \mathbf W_0\mathbf x + \frac{\alpha}{r}\,\mathbf B\mathbf A\mathbf x,\qquad \mathbf B \text{ startet bei } \mathbf 0.$$

**Ablauf in der Vorlesung:** 4.1 ergänzen und ausführen (Prüfungen ✅), dann die Trainingszelle 4.2 starten – und zurück zur
Vorlesung. Was LoRA genau tut, erarbeiten wir, während das Training läuft; in Aufgabe 5 rechnen wir es im Code nach.

### 4.1 Konfiguration

`CODE` (1 Zeile). Ergänzen Sie in `lora_modell` die **Zielmodule**: LoRA soll an die Query- und die Value-Projektion jeder
Attention-Schicht. Im Modell heißen sie `q_proj` und `v_proj` (siehe `print(basis)` in 1.1). Ist `module` beim Aufruf nicht angegeben
(`None`), soll die Liste `["q_proj", "v_proj"]` verwendet werden – in Python geht das kurz mit `module = module or [...]`.
**Erwartung:** `print_trainable_parameters()` meldet 460 800 trainierbare Parameter (0,34 %) – woher genau diese Zahl kommt, rechnen
Sie in Aufgabe 5 von Hand nach.

In [ ]:
abschnitt("Aufgabe 4")
def lora_modell(r=8, module=None):
    """Frisches Basismodell + LoRA-Adapter (W0 eingefroren, nur A und B trainierbar)."""
    torch.manual_seed(0)                                # gleiche Zufallszahlen fuer A bei jedem Aufruf
    m = AutoModelForCausalLM.from_pretrained(MODELL, torch_dtype=torch.float32).to(DEVICE)   # frische Kopie laden
    # TODO 4.1: Zielmodule: wenn module None ist, die Attention-Projektionen fuer Query und Value (Namen wie im Skript, Listing PEFT)
    raise NotImplementedError("TODO 4.1")
    # r = Rang des Updates (Engpassbreite), lora_alpha / r = Skalierung s des Updates (hier 16 / 8 = 2),
    # lora_dropout = Dropout auf dem LoRA-Pfad gegen Ueberanpassung bei nur 300 Beispielen.
    cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules=module, task_type="CAUSAL_LM")
    return get_peft_model(m, cfg)                       # PEFT baut die LoRA-Huellen ein und friert alles andere ein

modell = lora_modell()
modell.print_trainable_parameters()
n_train = sum(p.numel() for p in modell.parameters() if p.requires_grad)   # requires_grad = wird trainiert
check("4.1 trainierbare Parameter = 460 800 (Bsp. 5.4: 30 x (9 216 + 6 144))", n_train == 460_800,
      "q_proj und v_proj als Zielmodule, r = 8")
# Der lange Pfad fuehrt durch die PEFT-Huelle zur ersten Schicht: base_layer = das alte q_proj (W0), lora_B = Matrix B
check("4.1 Basismodell eingefroren (W0 ohne Gradient)",
      lambda: not modell.base_model.model.model.layers[0].self_attn.q_proj.base_layer.weight.requires_grad)
check("4.1 B startet bei 0 -> Modell ist zu Beginn exakt das vortrainierte",
      lambda: float(modell.base_model.model.model.layers[0].self_attn.q_proj.lora_B["default"].weight.detach().abs().max()) == 0.0)

### 4.2 Training

`AUSFÜHREN` · Skript Listing Training, Tab. LoRA-Hyperparameter · ≈ 2 Min. + **≈ 4–7 Min. Rechenzeit** (Lösungslauf: 0,8–1,4 s/Schritt
auf 4 Kernen eines Apple-M-Prozessors, 5,3 s/Schritt auf 2 Kernen x86)

**Konfiguration** (wie in der Vorlesung, vorab festgelegt):

| Größe | Wert | Bedeutung |
|---|---|---|
| Rang $r$, Skalierung $\alpha$ | 8, 16 | Engpassbreite; Update wird mit $\alpha/r = 2$ multipliziert |
| Dropout | 0,05 | zufälliges Abschalten auf dem LoRA-Pfad (Regularisierung) |
| Zielmodule | `q_proj`, `v_proj` | in allen 30 Schichten |
| Optimierer, Lernrate | AdamW, $2\cdot10^{-4}$ | |
| Batchgröße, Schritte | 4, **150** | $150\cdot4 = 600$ Beispiele $=$ 2 Epochen über 300 Beispiele |
| Verlust | nur Antwort-Token | Loss-Maske aus Aufgabe 3 |

Jeder Trainingsschritt besteht aus vier Teilen: **(1)** Vorwärtsrechnung mit Verlust, **(2)** alte Gradienten löschen
(`zero_grad`), **(3)** Gradienten berechnen (`backward`), **(4)** Parameter einen Schritt verändern (`step`). Der Validierungsverlust auf
`val.jsonl` wird alle 25 Schritte protokolliert – zur Beobachtung, **nicht** zur Modellauswahl (Skript Abschn. 4.2). Der Adapter wird
nach `results/adapter_stoermeldung/` gespeichert.

**Während das Training läuft,** geht die Vorlesung weiter. Die Interpretationsfragen unten beantworten Sie, sobald Kurve und
Prüfung ✅ erscheinen – spätestens zu Beginn von Aufgabe 5.

**Notbremse:** Dauert ein Schritt deutlich länger als 8 s (Ausgabe nach Schritt 10), abbrechen (■) und `NUTZE_REFERENZ_ADAPTER = True`
setzen. **Erwartung:** Val-Verlust 1,82 vor dem Training, 0,53 nach 150 Schritten; Adapter 1,9 MB.

In [ ]:
abschnitt("Aufgabe 4")
NUTZE_REFERENZ_ADAPTER = False          # True: mitgelieferten Adapter laden statt trainieren

SCHRITTE, BATCH, LR = 150, 4, 2e-4      # Trainingsschritte, Beispiele je Schritt, Lernrate

def batches(daten, batch_size, rng):
    """Mischt die Beispiele und liefert gepaddete Batches (Padding rechts, Labels -100 auf Padding)."""
    idx = rng.permutation(len(daten))                   # zufaellige Reihenfolge der Beispiele
    for i in range(0, len(idx) - batch_size + 1, batch_size):   # in Schritten von batch_size durchgehen
        bsp = [kodiere(daten[j]) for j in idx[i:i + batch_size]]   # input_ids und labels der Beispiele dieses Batches
        L = max(len(b["input_ids"]) for b in bsp)       # dynamisches Padding: nur bis zur laengsten Folge im Batch
        # Kuerzere Folgen rechts auffuellen: input_ids mit dem Padding-Token, labels mit -100 (dort wird nicht gelernt)
        ids = torch.tensor([b["input_ids"] + [tok.pad_token_id] * (L - len(b["input_ids"])) for b in bsp])
        lab = torch.tensor([b["labels"] + [-100] * (L - len(b["labels"])) for b in bsp])
        # attention_mask: 1 = echtes Token, 0 = Padding (das Modell soll Padding ignorieren)
        att = torch.tensor([[1] * len(b["input_ids"]) + [0] * (L - len(b["input_ids"])) for b in bsp])
        yield ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)   # yield: liefert einen Batch nach dem anderen

def val_verlust(m):
    """Mittlerer Verlust auf den 30 Validierungsbeispielen (ohne Training)."""
    m.eval(); tot, n = 0.0, 0                           # eval(): Dropout aus
    with torch.inference_mode():
        for ids, att, lab in batches(val, 5, np.random.default_rng(1)):
            tot += m(input_ids=ids, attention_mask=att, labels=lab).loss.item(); n += 1
    m.train(); return tot / n                           # zurueck in den Trainingsmodus

if NUTZE_REFERENZ_ADAPTER:
    # Fallback: frisches Basismodell laden und den mitgelieferten Adapter (A und B) daraufsetzen
    torch.manual_seed(0)
    basis_ref = AutoModelForCausalLM.from_pretrained(MODELL, torch_dtype=torch.float32).to(DEVICE)
    modell = PeftModel.from_pretrained(basis_ref, REFDIR / "adapter_stoermeldung")
    verlauf = lade_referenz("a4_training")["verlauf"]
    print("Referenz-Adapter geladen (kein Training).")
else:
    modell = lora_modell()                              # frisches Modell mit LoRA (B = 0)
    # AdamW bekommt NUR die trainierbaren Parameter, also die Matrizen A und B
    opt = torch.optim.AdamW([p for p in modell.parameters() if p.requires_grad], lr=LR)
    rng = np.random.default_rng(0); modell.train()     # train(): Dropout an
    verlauf = {"schritt": [], "train_loss": [], "val_schritt": [0], "val_loss": [val_verlust(modell)]}
    t0, schritt, lauf = time.perf_counter(), 0, []
    while schritt < SCHRITTE:                           # so lange, bis 150 Schritte erreicht sind (jede Runde = 1 Epoche)
        for ids, att, lab in batches(train, BATCH, rng):
            loss = modell(input_ids=ids, attention_mask=att, labels=lab).loss   # (1) vorwaerts; Shift + Cross-Entropy intern
            opt.zero_grad()                             # (2) alte Gradienten loeschen
            loss.backward()                             # (3) Gradienten fuer A und B berechnen
            opt.step()                                  # (4) A und B einen Schritt verbessern
            schritt += 1; lauf.append(loss.item())
            if schritt % 10 == 0:                       # alle 10 Schritte: Mittelwert ausgeben (% = Rest der Division)
                verlauf["schritt"].append(schritt); verlauf["train_loss"].append(float(np.mean(lauf))); lauf = []
                print(f"  Schritt {schritt:3d}: train loss {verlauf['train_loss'][-1]:.3f} | "
                      f"{(time.perf_counter() - t0) / schritt:.2f} s/Schritt")
            if schritt % 25 == 0:                       # alle 25 Schritte: Validierungsverlust (nur beobachten)
                verlauf["val_schritt"].append(schritt); verlauf["val_loss"].append(val_verlust(modell))
            if schritt == SCHRITTE:
                break
    dauer = time.perf_counter() - t0
    modell.save_pretrained(RESULTS / "adapter_stoermeldung")          # speichert nur A und B, nicht das Basismodell
    save_log("a4_training", verlauf=verlauf, dauer_s=dauer, schritte=SCHRITTE, batch=BATCH, lr=LR, r=8, alpha=16,
             trainierbar=n_train)
    print(f"Training: {dauer / 60:.1f} Min. ({dauer / SCHRITTE:.2f} s/Schritt)")

# Verlustkurve: Train (Mittel je 10 Schritte) und Val (alle 25 Schritte)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(verlauf["schritt"], verlauf["train_loss"], "o-", label="Train (Mittel je 10 Schritte)")
ax.plot(verlauf["val_schritt"], verlauf["val_loss"], "s--", label="Val")
ax.set_xlabel("Schritt"); ax.set_ylabel("Verlust (nur Antwort-Token)"); ax.grid(alpha=.3); ax.legend(); plt.show()
# Groesse aller Dateien im Adapter-Ordner zusammenzaehlen
groesse = sum(f.stat().st_size for f in (REFDIR if NUTZE_REFERENZ_ADAPTER else RESULTS).joinpath("adapter_stoermeldung").glob("*"))
print(f"Adapter auf der Platte: {groesse / 1e6:.1f} MB")
check("4 Verlust ist gesunken (letzter Train-Wert < halber Startwert der Val-Kurve)",
      lambda: verlauf["train_loss"][-1] < 0.5 * verlauf["val_loss"][0])

### Interpretation Aufgabe 4

`DENKEN`

1. Wie schnell fällt der Verlust in den ersten 20 Schritten – und warum startet das Modell trotz $\Delta\mathbf W=\mathbf 0$ nicht bei
   $\ln|\mathcal V|\approx10{,}8$ (dem Verlust eines Modells, das alle 49 152 Token gleich wahrscheinlich findet)? Denken Sie an das
   vortrainierte Modell.
2. Der Adapter ist rund 2 MB groß, das Basismodell 538 MB. Was heißt das für 20 Kundenwerke mit je eigener Anpassung?

**Ihre Antwort:**

## Aufgabe 5 · LoRA von innen

`PRINZIP` 5.1–5.4 · `CODE` 5.2, 5.3 (je 1 Zeile) · `PAPIER` 5.4 · `DENKEN` · Block nach der Folie „Zwischenstand: LoRA“ ·
Skript Def. 5.2, Bsp. 5.3, 5.4 · ≈ 13 Min.

**Lernziel** (Verstehen): die LoRA-Formel in Code übersetzen und zeigen, dass PEFT genau das rechnet.

**Die Formel** (Skript Def. 5.2). Eine lineare Schicht $\mathbf h=\mathbf W_0\mathbf x$ mit vortrainierter Matrix
$\mathbf W_0\in\mathbb R^{d\times k}$ wird ersetzt durch

$$
\mathbf h = \mathbf W_0\mathbf x + \Delta\mathbf W\mathbf x = \mathbf W_0\mathbf x + \frac{\alpha}{r}\,\mathbf B\mathbf A\mathbf x,
\qquad \mathbf B\in\mathbb R^{d\times r},\ \mathbf A\in\mathbb R^{r\times k},\ r\ll\min(d,k).
$$

$\mathbf W_0$ ist eingefroren; trainiert werden nur $\mathbf A$ (zufällig initialisiert) und $\mathbf B$ (startet bei $\mathbf 0$, also
ist das Modell zu Beginn exakt das vortrainierte). $s=\alpha/r$ skaliert das Update.

**Intuition.** Statt die große Matrix $\mathbf W_0$ zu verändern, lernt LoRA zwei schmale Matrizen: $\mathbf A$ drückt die Eingabe auf
$r$ Zahlen zusammen (Engpass), $\mathbf B$ bläst sie wieder auf. Das Produkt $\mathbf B\mathbf A$ hat die Form von $\mathbf W_0$, aber
höchstens Rang $r$. Alles in dieser Aufgabe ist Matrixrechnung in wenigen Zeilen.

### 5.1 Was heißt Rang?

`PRINZIP`. Ein äußeres Produkt $\mathbf u\mathbf v^\top$ (Spalte mal Zeile) ist eine Matrix vom Rang 1: Jede Zeile ist ein Vielfaches
von $\mathbf v^\top$. Eine Summe aus $r$ solchen Richtungspaaren hat höchstens Rang $r$. Die folgende 6 × 5-Matrix ist aus zwei
Richtungspaaren gebaut: Sie hat $d\cdot k = 30$ Einträge, wird aber von $r(d+k) = 2\cdot(6+5) = 22$ Zahlen vollständig beschrieben.
Bei großen Matrizen wird das Verhältnis drastisch.

In [ ]:
abschnitt("Aufgabe 5")
u1, v1 = np.array([1., 0, 2, 0, 1, 1]), np.array([1., 1, 0, 0, 2])   # erstes Richtungspaar (u: 6 Zahlen, v: 5 Zahlen)
u2, v2 = np.array([0., 1, 0, 1, 0, 2]), np.array([0., 1, 1, 1, 0])   # zweites Richtungspaar
M = np.outer(u1, v1) + np.outer(u2, v2)                 # np.outer(u, v) = u v^T; Summe zweier Rang-1-Matrizen
print(M)
print("Form:", M.shape, "| Rang:", np.linalg.matrix_rank(M))
print("Eintraege d*k:", M.size, "| Zahlen in den Richtungspaaren r*(d+k):", 2 * (6 + 5))

d, k, r = 576, 576, 8                                   # W_q von SmolLM2, LoRA-Rang 8
print(f"\nW_q: d*k = {d * k:,} Eintraege | LoRA r*(d+k) = {r * (d + k):,} = {r * (d + k) / (d * k):.1%}")

### 5.2 Ein Rang-1-Update von Hand (Bsp. 5.3)

`CODE` (1 Zeile). Rechenbeispiel mit $d=k=3$, $r=1$ (Skalierung hier weggelassen):

$$
\mathbf x=\begin{pmatrix}2\\5\\1\end{pmatrix},\quad
\mathbf A=\begin{pmatrix}1 & 0 & -1\end{pmatrix},\quad
\mathbf B=\begin{pmatrix}0{,}5\\0\\1\end{pmatrix},\qquad
\mathbf A\mathbf x = 2-1 = 1,\qquad
\mathbf B(\mathbf A\mathbf x) = \begin{pmatrix}0{,}5\\0\\1\end{pmatrix}.
$$

Erst der Engpass $\mathbf A\mathbf x$ (eine einzige Zahl), dann zurück mit $\mathbf B$. Ergänzen Sie die Zeile für `update`
(Matrixprodukt mit `@`). **Erwartung:** $\mathbf A\mathbf x = 1$, Update $= (0{,}5;\ 0;\ 1)$ – identisch mit
$\Delta\mathbf W\mathbf x = (\mathbf B\mathbf A)\mathbf x$, aber mit 6 statt 9 gespeicherten Zahlen.

In [ ]:
x = np.array([2., 5., 1.])
A = np.array([[1., 0., -1.]])                           # 1 x 3: auf welche Eingaberichtung reagiert das Update?
B = np.array([[0.5], [0.], [1.]])                       # 3 x 1: in welche Ausgaberichtung wirkt es?
engpass = A @ x                                         # r = 1 Zahl
# TODO 5.2: Update = B mal Engpass (Matrixprodukt @)
raise NotImplementedError("TODO 5.2")
print("A x       =", engpass)
print("B (A x)   =", update)
dW = B @ A                                              # die volle 3 x 3-Matrix - LoRA speichert sie nie
print("Delta W   =\n", dW)
print("Delta W x =", dW @ x)
check("5.2 Rang-1-Update wie Bsp. 5.3", lambda: np.allclose(update, [0.5, 0, 1]))

### 5.3 Eine LoRA-Schicht selbst bauen

`CODE` (1 Zeile) + `PRINZIP`. Eine eigene Klasse mit der eingefrorenen Matrix $\mathbf W_0$ = `q_proj` aus Schicht 0 von SmolLM2,
$\mathbf A$ zufällig, $\mathbf B = \mathbf 0$, Skalierung $s = \alpha/r$. Ergänzen Sie `forward` nach der Formel oben.

**Zeilen-Schreibweise:** In PyTorch steht jedes Token als *Zeile* in `x` (Form: Token × $k$). Aus $\mathbf W\mathbf x$ wird dann
`x @ W.T`. Die LoRA-Formel lautet in dieser Schreibweise

$$\texttt{h} = \texttt{x @ W0.T} + s\cdot(\texttt{x @ A.T @ B.T}).$$

Danach prüft die Zelle drei Aussagen: **(1)** Zu Beginn ist die Schicht exakt die vortrainierte. **(2)** Im ersten Schritt bekommt nur
$\mathbf B$ einen Gradienten. **(3)** Nach dem Einrechnen (*merge*) $\mathbf W = \mathbf W_0 + s\,\mathbf B\mathbf A$ liefert eine einzige
Matrix dieselbe Ausgabe.

In [ ]:
class LoRALinear(torch.nn.Module):
    """Eine lineare Schicht mit LoRA: h = W0 x + s * B A x (Skript Def. 5.2)."""
    def __init__(self, W0, r, alpha):
        super().__init__()                                      # Pflichtzeile fuer eigene PyTorch-Module
        d, k = W0.shape                                         # Ausgabe- und Eingabebreite
        self.W0 = W0                                            # vortrainiert, eingefroren (kein Parameter -> kein Gradient)
        self.A = torch.nn.Parameter(torch.randn(r, k) * 0.01)   # r x k, zufaellig; Parameter = wird trainiert
        self.B = torch.nn.Parameter(torch.zeros(d, r))          # d x r, Null -> Start = vortrainierte Schicht
        self.s = alpha / r                                      # Skalierung des Updates

    def forward(self, x):
        """Wird bei lora(x) aufgerufen. x: (Token x k) -> Ergebnis: (Token x d)."""
        # TODO 5.3: h = W0 x + s * B A x  (Zeilen-Schreibweise: x @ W0.T + ...)
        raise NotImplementedError("TODO 5.3")

torch.manual_seed(0)
q_proj_0 = basis.model.layers[0].self_attn.q_proj               # die Query-Projektion der ersten Schicht
W0 = q_proj_0.weight.detach()                                   # 576 x 576, aus dem vortrainierten Modell (ohne Gradient)
lora = LoRALinear(W0, r=8, alpha=16)
print("trainierbar:", sum(p.numel() for p in lora.parameters()), "Zahlen statt", W0.numel())

x = torch.randn(4, 576)                                         # 4 zufaellige "Token" als Eingabe
with torch.no_grad():
    start_gleich = torch.allclose(lora(x), q_proj_0(x))         # allclose: gleich bis auf Rundungsfehler?
print("(1) Start = vortrainierte Schicht:", start_gleich)

ziel = torch.randn(4, 576)                                      # irgendein Lernziel
verlust = ((lora(x) - ziel) ** 2).mean()                        # mittlerer quadratischer Fehler
verlust.backward()                                              # Gradienten fuer A und B berechnen
grad_A, grad_B = float(lora.A.grad.abs().max()), float(lora.B.grad.abs().max())   # groesster Betrag je Gradient
print(f"(2) groesster Gradient im ersten Schritt: A {grad_A:.4f} | B {grad_B:.4f}")

with torch.no_grad():                                           # ein Schritt Gradientenabstieg von Hand
    lora.A -= 0.1 * lora.A.grad                                 # Parameter = Parameter - Lernrate * Gradient
    lora.B -= 0.1 * lora.B.grad
    W_merged = W0 + lora.s * lora.B @ lora.A                    # einrechnen: wieder eine einzige 576 x 576-Matrix
    merge_gleich = torch.allclose(lora(x), x @ W_merged.T, atol=1e-4)
print("(3) eingerechnete Matrix = LoRA-Schicht:", merge_gleich)
check("5.3 LoRA-Schicht: Start gleich, nur B mit Gradient, merge gleich",
      lambda: start_gleich and grad_A == 0.0 and grad_B > 0 and merge_gleich)

### 5.4 PEFT rechnet genau das

`PAPIER` + `PRINZIP`. Erst die Parameterzahl von Hand (Bsp. 5.4). Eine LoRA-Schicht für eine Matrix der Form $d\times k$ hat

$$\underbrace{r\cdot k}_{\mathbf A} + \underbrace{d\cdot r}_{\mathbf B} = r\,(d+k)\ \text{ trainierbare Parameter.}$$

SmolLM2 hat 30 Schichten, $\mathbf W_q \in \mathbb R^{576\times576}$ und – wegen Grouped-Query Attention – $\mathbf W_v \in
\mathbb R^{192\times576}$. Wie viele LoRA-Parameter ergeben sich für $r=8$? Welcher Anteil von 134,5 Mio. ist das? Tragen Sie Ihre
Rechnung in die Tabelle ein. Danach sieht die Code-Zelle im trainierten Modell aus Aufgabe 4 nach: Formen von $\mathbf A$ und
$\mathbf B$, Skalierung, Summe über alle Schichten – und ob die PEFT-Schicht dasselbe ausgibt wie die Formel.
**Erwartung:** 460 800 Parameter; $\Delta\mathbf W$ hat Rang 8 und ist ≈ 0,6 % so groß wie $\mathbf W_0$ (gemessen an der Norm).

**Ihre Papierrechnung 5.4** (Doppelklick, dann ausfüllen)

| Matrix | d × k | r(d + k) |
|---|---|---|
| `q_proj` | | |
| `v_proj` | | |
| je Schicht | | |
| 30 Schichten, Anteil | | |

In [ ]:
# Im trainierten Modell nachsehen: PEFT hat das alte q_proj in eine Huelle gepackt
q = modell.base_model.model.model.layers[0].self_attn.q_proj   # Schicht 0, Query-Projektion
W0 = q.base_layer.weight                           # das alte q_proj: eingefroren
A = q.lora_A["default"].weight                     # r x k
B = q.lora_B["default"].weight                     # d x r
s = q.scaling["default"]                           # alpha / r
print("W0", tuple(W0.shape), "| A", tuple(A.shape), "| B", tuple(B.shape), "| s = alpha/r =", s)

# alle Parameter zaehlen, deren Name "lora_" enthaelt (alle A- und B-Matrizen in allen 30 Schichten)
anzahl = sum(p.numel() for name, p in modell.named_parameters() if "lora_" in name)
print(f"LoRA-Parameter im ganzen Modell: {anzahl:,}")

modell.eval()                                      # Dropout aus
x = torch.randn(4, 576, device=W0.device)
with torch.no_grad():
    h_peft = q(x)                                  # was PEFT rechnet
    h_formel = x @ W0.T + s * (x @ A.T @ B.T)      # Def. 5.2, von Hand (Zeilen-Schreibweise wie in 5.3)
    dW = s * B @ A                                 # die Aenderung, die LoRA an W0 vornimmt
    # Norm = "Groesse" einer Matrix (Wurzel aus der Summe aller quadrierten Eintraege)
    norm_B, rang_dW, anteil = float(B.norm()), int(torch.linalg.matrix_rank(dW)), float(dW.norm() / W0.norm())
print("PEFT-Ausgabe = Formel:", torch.allclose(h_peft, h_formel, atol=1e-5))
print(f"Nach dem Training: Norm von B = {norm_B:.3f} (Start: 0) | Rang von Delta W = {rang_dW} | Norm Delta W / Norm W0 = {anteil:.2%}")
check("5.4 460 800 LoRA-Parameter (Bsp. 5.4)", anzahl == 460_800)
check("5.4 PEFT rechnet h = W0 x + s B A x", lambda: torch.allclose(h_peft, h_formel, atol=1e-5))

### Interpretation Aufgabe 5

`DENKEN`

1. Warum bekommt $\mathbf A$ im ersten Schritt keinen Gradienten (5.3, Ausgabe (2))? Was passierte, wenn man auch $\mathbf A$ mit 0
   startete? (Tipp: Der Gradient von $\mathbf A$ ist $\partial L/\partial\mathbf A = s\,(\mathbf B^\top\mathbf u)\,\mathbf x^\top$ mit
   $\mathbf u = \partial L/\partial\mathbf h$, der von $\mathbf B$ ist $s\,\mathbf u\,(\mathbf A\mathbf x)^\top$.)
2. $\Delta\mathbf W$ ist nach dem Training nur rund 0,6 % so groß wie $\mathbf W_0$ (gemessen an der Norm) und hat Rang 8. Was sagt das
   über die Hypothese von LoRA, dass die Änderung $\Delta\mathbf W$ bei der Anpassung einen niedrigen „intrinsischen Rang“ hat?
3. Nach dem Einrechnen gibt es keine Zusatzlatenz (5.3, Ausgabe (3)). Warum hält man Adapter im Betrieb trotzdem oft getrennt?

**Ihre Antwort:**

## Aufgabe 6 · Evaluation vorher und nachher

`AUSFÜHREN` 6.1 · `CODE` 6.2 (2 Zeilen) · `DENKEN` 6.3 · direkt im Anschluss an Aufgabe 5 · Skript Abschn. 7.6 (Bsp. gepaarter
Vergleich), 8.1–8.2 · ≈ 9 Min. + ≈ 1 Min. Rechenzeit

**Lernziel** (Analysieren): eine Verbesserung mit Unsicherheit berichten und gepaart prüfen, statt zwei Prozentzahlen
nebeneinanderzulegen.

### 6.1 Metriken und Intervall

`AUSFÜHREN`. Dieselben 50 Eval-Meldungen mit Adapter (zero-shot, greedy) bewerten und alle drei Varianten in einer Tabelle
vergleichen. Eine Quote $\hat p$ auf nur $n$ Beispielen schwankt; ihr Standardfehler und das 95-%-Intervall sind (Skript Gl. SE)

$$\mathrm{SE}(\hat p) = \sqrt{\frac{\hat p\,(1-\hat p)}{n}},\qquad \text{95-\%-Intervall} \approx \hat p \pm 1{,}96\cdot\mathrm{SE}.$$

Beispiel: $\hat p = 0{,}70$, $n = 50$ ⇒ $\mathrm{SE} = 0{,}065$, Intervall ≈ 57–83 %. Die Funktion `intervall` aus den Werkzeugen
rechnet genau das.

**Erwartung** (Lösungslauf): nach LoRA 100 % Format, 88 % Schema, 66 % `anlage`, 74 % `messwert` exakt, **56 % komplett**
(95-%-Intervall 42–70 %).

In [ ]:
abschnitt("Aufgabe 6")
t0 = time.perf_counter()
aus_lora = generiere(modell, tok, [chat(b["meldung"]) for b in ev])    # das trainierte Modell, zero-shot
print(f"Generierung 50 Meldungen mit Adapter: {time.perf_counter() - t0:.0f} s")
q_lora, bew_lora = auswerten(aus_lora, ev, "a5_lora")
# Tabelle: eine Zeile je Variante, eine Spalte je Stufe
tab = pd.DataFrame({"zero-shot": q_zero, "few-shot (3)": q_few, "LoRA, zero-shot": q_lora}).T
display(tab.style.format("{:.0%}"))
lo, hi = intervall(q_lora["komplett"], len(ev))         # 95-%-Intervall fuer "komplett" nach LoRA
print(f"komplett nach LoRA: {q_lora['komplett']:.0%}, 95-%-Intervall [{lo:.0%}; {hi:.0%}] bei n = {len(ev)}")

### 6.2 Gepaarter Vorzeichentest

`CODE` (2 Zeilen). Vorher und nachher wurden auf **denselben** 50 Meldungen gemessen. Entscheidend sind nur die Meldungen, deren
Ergebnis sich ändert: $b$ = besser geworden (vorher falsch, nachher richtig), $c$ = schlechter geworden. Unter der Nullhypothese
„kein Unterschied“ ist jede der $n=b+c$ Änderungen wie ein fairer Münzwurf mit Wahrscheinlichkeit ½ eine Verbesserung. Die
Wahrscheinlichkeit für ein mindestens so extremes Ergebnis in eine der beiden Richtungen ist (exakter Vorzeichentest bzw. McNemar,
Skript Bsp. gepaarter Vergleich):

$$p = \min\Bigl(1,\; 2\sum_{k=0}^{\min(b,c)}\binom{n}{k}\Bigl(\frac12\Bigr)^{n}\Bigr).$$

In Python: $\binom{n}{k}$ = `math.comb(n, k)`, $(\tfrac12)^n$ = `1 / 2**n`, die Summe z. B. mit `sum(... for k in range(min(b, c) + 1))`.
**Kontrollwert:** $b=12$, $c=3$ ⇒ $p = 2\cdot(1+15+105+455)/32\,768 \approx 0{,}035$.
**Erwartung** (Lösungslauf): LoRA gegen few-shot 21 besser, 4 schlechter, $p \approx 0{,}001$.

In [ ]:
def vorzeichentest(b, c):
    """Exakter zweiseitiger Vorzeichentest (McNemar exakt) fuer b Verbesserungen und c Verschlechterungen."""
    n = b + c                                           # nur die Meldungen, die sich veraendert haben
    if n == 0:
        return 1.0                                      # keine Aenderung -> kein Hinweis auf einen Unterschied
    # TODO 6.2: p = 2 * Summe ueber k = 0 .. min(b, c) von math.comb(n, k) / 2**n, hoechstens 1
    raise NotImplementedError("TODO 6.2")

check("6.2 Kontrollwert b = 12, c = 3 -> p = 0,035", lambda: abs(vorzeichentest(12, 3) - 0.0352) < 1e-3)
check("6.2 symmetrisch und hoechstens 1", lambda: vorzeichentest(3, 12) == vorzeichentest(12, 3) and vorzeichentest(5, 5) == 1.0)

ergebnisse = {}
for name, bew_vor in [("zero-shot", bew_zero), ("few-shot", bew_few)]:
    # Meldung fuer Meldung vergleichen: v = Bewertung vorher, n_ = Bewertung nach LoRA
    b = sum((not v["komplett"]) and n_["komplett"] for v, n_ in zip(bew_vor, bew_lora))   # vorher falsch, nachher richtig
    c = sum(v["komplett"] and not n_["komplett"] for v, n_ in zip(bew_vor, bew_lora))     # vorher richtig, nachher falsch
    ergebnisse[name] = {"besser": int(b), "schlechter": int(c), "p": vorzeichentest(b, c)}
    print(f"LoRA gegen {name:9}: {b:2d} besser, {c:2d} schlechter, p = {ergebnisse[name]['p']:.2g}")
save_log("a5_vergleich", quoten={"zero": q_zero, "few": q_few, "lora": q_lora}, vorzeichentest=ergebnisse,
         intervall_komplett_lora=[lo, hi])

### 6.3 Fehler lesen

`AUSFÜHREN`, dann `DENKEN`. Erst lesen, dann an Hyperparametern drehen (Skript Abschn. 8.2): Welche Meldungen sind nach dem
Training noch nicht *komplett* richtig – und gehören sie zu einer bestimmten Art (typisch / fehlend / abk)? Ordnen Sie die Fehler
Kategorien zu, z. B. Format- oder Schemafehler, falscher Wert aus der Meldung übernommen, Abkürzung nicht aufgelöst, erfundener Wert
(Halluzination), fehlende Angabe nicht als `null` gesetzt (Skript Tab. Fehlerkategorien).
**Erwartung** (Lösungslauf): Alle Fehler liegen bei *typisch* und *abk*.

In [ ]:
# Liste der Meldungen, die nach LoRA nicht "komplett" sind: (Beispiel, Modellausgabe, Bewertung)
fehler = [(b, a, e) for b, a, e in zip(ev, aus_lora, bew_lora) if not e["komplett"]]
print(f"{len(fehler)} von {len(ev)} Meldungen nicht komplett richtig; nach Art:", dict(Counter(b["art"] for b, _, _ in fehler)))
for b, a, e in fehler[:6]:                              # die ersten sechs Fehler zum Lesen
    print(f"\n[{b['id']}, {b['art']}] {b['meldung']}\n  Ziel   : {ziel_json(b['ziel'])}\n  Modell : {a[:200]}")
# Quote "komplett" getrennt nach Art der Meldung
quote_art = {art: float(np.mean([e["komplett"] for b, e in zip(ev, bew_lora) if b["art"] == art])) for art in ("typisch", "fehlend", "abk")}
print("\nkomplett nach Art:", {k: f"{v:.0%}" for k, v in quote_art.items()})
save_log("a5_fehler", quote_art=quote_art, fehler_ids=[b["id"] for b, _, _ in fehler])

### Interpretation Aufgabe 6

`DENKEN`

1. Berichten Sie das Ergebnis in **einem** Satz so, wie es in eine Eval-Karte gehört: Wert, $n$, Intervall, gepaarter Vergleich.
2. Ist die Verbesserung gegenüber zero-shot gesichert? Wäre sie es auch mit nur 10 Eval-Meldungen und denselben Anteilen? (Rechnen
   Sie mit `vorzeichentest` nach: $b$ und $c$ entsprechend herunterskalieren.)
3. Welche Fehlerkategorie dominiert in 6.3 – und welche Maßnahme folgt daraus?

**Ihre Antwort:**

## Aufgabe 7 · Regressions-Set: Hat das Modell etwas verlernt?

`AUSFÜHREN` · `DENKEN` · **Nachbereitung** · Skript Abschn. 4.4 (Vergessen und Alignment-Steuer), Eval-Karte (Regressions-Set) ·
≈ 4 Min. + ≈ 1 Min. Rechenzeit

Zehn allgemeine Anweisungen **ohne** Extraktions-Systemprompt, vor und nach dem Training. Gemessen wird grob, wie oft das Modell mit
einem JSON-Objekt antwortet (Antwort beginnt mit `{`), obwohl niemand danach gefragt hat; dazu lesen Sie die Antworten.
**Erwartung:** in beiden Fällen 0 % JSON – aber lesen Sie, ob sich die Antworten verändert haben (das Basismodell mit 135 Mio.
Parametern ist auch vorher schwach).

In [ ]:
abschnitt("Aufgabe 7")
chats_r = [[{"role": "user", "content": r["anweisung"]}] for r in regr]   # nur die Anweisung, kein Systemprompt
r_vor = generiere(basis, tok, chats_r, max_new_tokens=80)                 # vorher: Basismodell
r_nach = generiere(modell, tok, chats_r, max_new_tokens=80)               # nachher: mit LoRA-Adapter
json_quote = lambda aus: float(np.mean([a.lstrip().startswith("{") for a in aus]))   # Anteil der Antworten, die mit { beginnen
print(f"Antwort beginnt mit '{{': vorher {json_quote(r_vor):.0%} | nachher {json_quote(r_nach):.0%}")
display(pd.DataFrame({"Anweisung": [r["anweisung"] for r in regr],
                      "vorher": [a[:110] for a in r_vor], "nachher (LoRA)": [a[:110] for a in r_nach]}))
save_log("a6_regression", json_vorher=json_quote(r_vor), json_nachher=json_quote(r_nach), vorher=r_vor, nachher=r_nach)

### Interpretation Aufgabe 7

`DENKEN`

1. Hat das Modell allgemeine Fähigkeiten verloren? Woran sehen Sie es – und ist das für den Einsatzzweck „nur Störmeldungen“
   überhaupt ein Problem?
2. Nennen Sie zwei Maßnahmen aus dem Skript, falls das Modell weiterhin allgemein nutzbar bleiben soll.

**Ihre Antwort:**

## Optional · Selbststudium

### O1 · Judge-Kosten und Bewertungsanweisung

`DENKEN` + `CODE`. Die Stufen „symptom/ursache/massnahme inhaltlich richtig“ bräuchten einen LLM-as-a-Judge (Skript Abschn. 7.5).
Die Kosten eines Eval-Laufs sind (Skript Gl. Judge-Kosten)

$$\text{Kosten} = n_{\text{Urteile}}\cdot\bigl(t_{\text{ein}}\cdot p_{\text{ein}} + t_{\text{aus}}\cdot p_{\text{aus}}\bigr)$$

mit der mittleren Zahl von Eingabe- und Ausgabe-Token $t$ und den Preisen $p$ je Token. Berechnen Sie die Kosten für 50 Vergleiche
**in beiden Reihenfolgen** ($n_{\text{Urteile}} = 100$, gegen Positions-Bias) bei 600 Eingabe- und 120 Ausgabe-Token, 2,50 € bzw.
10 € je 1 Mio. Token – und für eine Hyperparametersuche mit 40 Läufen. Formulieren Sie danach eine Bewertungsanweisung (Rubrik) mit
Referenz. **Erwartung:** 0,27 € je Lauf, 10,80 € für 40 Läufe.

In [ ]:
abschnitt("Optional O1")
n_urteile, t_ein, t_aus, p_ein, p_aus = 2 * 50, 600, 120, 2.50e-6, 10e-6   # Preise in € je Token (2,50 € / 1 Mio.)
# TODO O1: Kosten je Lauf = n_urteile * (t_ein * p_ein + t_aus * p_aus); dann mal 40
raise NotImplementedError("TODO O1")
print(f"Judge-Kosten: {kosten_lauf:.2f} € je Lauf, {kosten_suche:.2f} € fuer 40 Laeufe")
check("O1 Kosten je Lauf = 0,27 €", lambda: abs(kosten_lauf - 0.27) < 1e-9)
# Beispiel einer Rubrik: {referenz} und {ausgabe} sind Platzhalter, die .format(...) unten einsetzt
RUBRIK = """Du bewertest, ob die Felder symptom, ursache und massnahme inhaltlich zur Stoermeldung passen.
Referenz: {referenz}
Ausgabe: {ausgabe}
Antworte je Feld mit RICHTIG oder FALSCH und einer Begruendung in einem Satz."""
print(RUBRIK.format(referenz=ziel_json(ev[0]["ziel"]), ausgabe=aus_lora[0]))

### O2 · Alle linearen Schichten statt nur `q_proj`/`v_proj`

`AUSFÜHREN` · ≈ 10 Min. Rechenzeit. Skript Abschn. 5.6: Mehrere Matrizentypen mit kleinem Rang helfen oft mehr als ein großer Rang.
Trainieren Sie mit allen sieben linearen Matrizen ($r=8$; Bsp. 5.4: 2,44 Mio. Parameter, 1,8 %) und vergleichen Sie *komplett* und
*streng* mit Aufgabe 6. **Achtung:** Die Wahl auf dem Eval-Set zu treffen wäre unsauber – hier ist es eine Diagnose, keine
Modellauswahl.

In [ ]:
abschnitt("Optional O2")
O2_AUSFUEHREN = False                   # True setzen, wenn Sie ≈ 10 Min. Rechenzeit uebrig haben
if O2_AUSFUEHREN:
    alle = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]   # alle 7 Matrizen je Schicht
    m2 = lora_modell(module=alle); m2.print_trainable_parameters()
    opt2 = torch.optim.AdamW([p for p in m2.parameters() if p.requires_grad], lr=LR)
    rng2, s2 = np.random.default_rng(0), 0; m2.train()
    while s2 < SCHRITTE:                # dieselbe Trainingsschleife wie in 4.2, nur ohne Ausgaben
        for ids, att, lab in batches(train, BATCH, rng2):
            l2 = m2(input_ids=ids, attention_mask=att, labels=lab).loss
            opt2.zero_grad(); l2.backward(); opt2.step(); s2 += 1
            if s2 == SCHRITTE: break
    q_alle, _ = auswerten(generiere(m2, tok, [chat(b["meldung"]) for b in ev]), ev, "o2_alle_module")
    display(pd.DataFrame({"q/v, r = 8": q_lora, "alle 7, r = 8": q_alle}).T.style.format("{:.0%}"))

## Laufzeiten und Prüfungen

`AUSFÜHREN` · Reine Rechenzeit je Aufgabe und Übersicht aller Prüfungen.

In [ ]:
abschnitt("Abschluss")
display(pd.DataFrame({"Rechenzeit (s)": RUNTIME}).round(1))       # gemessene Rechenzeit je Aufgabe
pflicht = sum(v for k, v in RUNTIME.items() if k.startswith("Aufgabe"))
print(f"Pflichtteil (Aufgaben 0-7): {pflicht / 60:.1f} Min. | Threads: {torch.get_num_threads()}")
save_log("laufzeiten", runtime_s=RUNTIME, pflicht_min=pflicht / 60, cpu_kerne=os.cpu_count())
print(f"Pruefungen: {sum(CHECKS.values())} ✅ / {sum(not v for v in CHECKS.values())} ❌")
for name, ok in CHECKS.items():                         # alle nicht bestandenen Pruefungen auflisten
    if not ok:
        print("  ❌", name)